# Sionna 0.19 – Main Ray Tracing Simulation
**Kernel:** `sionna019` · Python 3.10 · Sionna 0.19.2 · TensorFlow 2.15

Migrated from Untitled(1).ipynb (Sionna 2.0) to Sionna 0.19.2 API.
Uses `scene.compute_paths()` and `scene.coverage_map()` (not PathSolver/RadioMapSolver).
GPS / UTM transforms, DEM lookup, TX/RX CSV loading all preserved from original.

## CELL 0 · Environment Setup & Imports

In [ ]:
import os, sys

import sys, json, csv, time, warnings, glob, re

# ── CPU/GPU mode — must be set before TF and Mitsuba imports ─────────────────
FORCE_CPU_RT = True   # True = CPU ray tracing (no VRAM limit); False = GPU
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
from scipy import stats
from scipy.stats import spearmanr
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except ImportError:
    pass

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if FORCE_CPU_RT:
    # Hide ALL GPUs from TensorFlow — fibonacci_lattice() returns TF tensors;
    # if TF places them on GPU, DrJit CPU (llvm_ad_rgb) cannot consume them
    # via DLPack → "Cannot create Dr.Jit CPU array from DLPack GPU tensor!"
    tf.config.set_visible_devices([], 'GPU')
    print(f'TF GPU  : hidden (FORCE_CPU_RT=True) — TF tensors stay on CPU')
elif gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'TF GPU  : {gpus[0].name}')
else:
    print('TF GPU  : NOT detected – running on CPU')
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)

import sionna
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver

_HAS_OFDM = False
try:
    from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies
    _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel)')
except (ImportError, AttributeError):
    try:
        from sionna.channel.ofdm import cir_to_ofdm_channel, subcarrier_frequencies
        _HAS_OFDM = True; print('OFDM    : OK  (sionna.channel.ofdm)')
    except: print('OFDM    : NOT found – power fallback will be used')

# ── Mitsuba – set variant BEFORE importing sionna ────────────────────────────
# Sionna 0.19 picks up whatever mi.set_variant() is active at import time.
# FORCE_CPU_RT=True → llvm_ad_rgb (CPU, no VRAM limit, works for full 81k-bld scene)
# FORCE_CPU_RT=False → cuda_ad_rgb (GPU, faster but OOMs on large BVH)
#
# IMPORTANT: this block must run before `import sionna` above.
# Change FORCE_CPU_RT here AND in CELL 0c — they must match.
_HAS_MI = False
try:
    import mitsuba as mi
    if FORCE_CPU_RT:
        _MI_VARIANTS = ['llvm_ad_rgb', 'scalar_rgb']
    else:
        _MI_VARIANTS = ['cuda_ad_rgb', 'llvm_ad_rgb', 'scalar_rgb']
    for _var in _MI_VARIANTS:
        try:
            mi.set_variant(_var)
            _HAS_MI = True
            print(f'Mitsuba : {mi.variant()}  (FORCE_CPU_RT={FORCE_CPU_RT})')
            break
        except Exception:
            continue
    if not _HAS_MI:
        print(f'Mitsuba : imported but no usable variant found')
except ImportError:
    print('Mitsuba : NOT installed')

_HAS_RIO = False
try:
    import rasterio as rio; _HAS_RIO = True; print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available')

_HAS_OSM = False
try:
    import osmnx as ox
    from shapely.geometry import box, Polygon, MultiPolygon
    from shapely.ops import unary_union
    _HAS_OSM = True; print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available – pip install osmnx shapely')

print(f'Python  : {sys.version.split()[0]}')
print(f'TF      : {tf.__version__}')
print(f'Sionna  : {sionna.__version__}')

def _safe(v):
    if hasattr(v, 'numpy'): return float(v.numpy())
    if hasattr(v, 'item'):  return float(v.item())
    return float(v)

def _to_numpy(t):
    if isinstance(t, tuple): return t[0].numpy() + 1j * t[1].numpy()
    if hasattr(t, 'numpy'): return t.numpy()
    return np.array(t)

def _cm_to_numpy(cm_obj):
    for attr in ('path_gain', 'rss', 'as_tensor'):
        if not hasattr(cm_obj, attr): continue
        val = getattr(cm_obj, attr)
        arr = val() if callable(val) else val
        if hasattr(arr, 'numpy'): return arr.numpy()
        try: return np.array(arr)
        except: pass
    raise AttributeError('Cannot extract path_gain from CoverageMap.')

## CELL 0b · Install Missing Dependencies

Run once if packages are missing.

In [ ]:
# Uncomment and run once, then restart kernel
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
#     'osmnx', 'shapely', 'pyvista', 'open3d', 'rasterio', 'pyproj',
#     'ipyleaflet', 'ipyvolume', 'tqdm'])
# print('Done – restart kernel.')

## CELL 0c · City Bounding Box

**Edit this cell to switch cities.** Keep the OSM download area ≤ 4 km × 4 km for speed.

The full Nottingham scene points you provided span ~17 km × 9 km (≈93k buildings).
A cropped city-centre tile is used for OSM download; full bounds are kept for Sionna scene.

In [ ]:
# ── City selector ─────────────────────────────────────────────────────────────
CITY_NAME = 'Nottingham'

# ── Full scene bbox (from sionna_web scene points) ────────────────────────────
# Point 0: (-1.447449, 52.918218)
# Point 1: (-1.447449, 53.001149)
# Point 2: (-1.206779, 53.001149)
# Point 3: (-1.206779, 52.918218)
SCENE_WEST   = -1.447449
SCENE_EAST   = -1.206779
SCENE_SOUTH  =  52.918218
SCENE_NORTH  =  53.001149

# Active scene bounds (full area)
WEST, EAST, SOUTH, NORTH = SCENE_WEST, SCENE_EAST, SCENE_SOUTH, SCENE_NORTH

center_lon = (WEST  + EAST)  / 2
center_lat = (SOUTH + NORTH) / 2

# ── Coordinate system for UK (zone 30N) ──────────────────────────────────────
UTM_EPSG = 32630   # WGS84 / UTM zone 30N  (UK)
BNG_EPSG = 27700   # British National Grid  (for UK DEM TIFFs)

# ── Project paths ─────────────────────────────────────────────────────────────
BASE_DIR  = os.path.expanduser(f'~/Documents/FYP2026/{CITY_NAME.lower()}')
OUT_DIR   = os.path.join(BASE_DIR, 'results_sionna019')
SCENE_DIR = BASE_DIR
os.makedirs(OUT_DIR, exist_ok=True)

# ── Scene rebuild control ────────────────────────────────────────────────────
# Set False to skip CELL 2+3 and load existing scene.xml directly in CELL 4
FORCE_REBUILD_SCENE = False

# DEM TIF – Nottingham terrain
DEM_TIFF    = '/home/georgeskai/Documents/Region/nottingham3602/uk_terrain_nottingham_aoi.tif'

SCENE_XML   = os.path.join(SCENE_DIR, 'scene', 'scene.xml')
MEASUREMENT_CSV = '/home/georgeskai/Documents/Region/nottingham3602/measurements_with_pathloss.csv'
TX_CSV      = os.path.join(SCENE_DIR, 'transmitter_positions.csv')
RX_CSV      = os.path.join(SCENE_DIR, 'receiver_locations.csv')
PARAMS_JSON = os.path.join(SCENE_DIR, 'scene_parameters.json')

# ── City-specific building height cap ───────────────────────────────────────
# Adjust per city. Used to reject SRTM/nDSM tree noise.
# Typical values: Nottingham=40, London=200, Paris=150, NYC=400
CITY_MAX_HEIGHT_M = 40.0
CITY_MIN_HEIGHT_M =  2.0

# ── RF link budget (match to Ofcom drive test metadata) ──────────────────────
# ── RF link budget — Ofcom Nottingham drive-test calibration ─────────────────
# Site: Nottingham | Date: 04.08.2016 | Ref: Ofcom drive-test dataset
FREQUENCY_HZ     = 3602.5e6  # 3602.5 MHz (Ofcom measured frequency)
BANDWIDTH_HZ     = 20e6      # channel bandwidth (Hz)

# TX site parameters (Ofcom spec exact values)
TX_LON           = -1.2559   # site longitude
TX_LAT           =  52.9863  # site latitude
TX_POWER_DBM     =  47.8     # amplifier output power (dBm)
TX_ANTENNA_GAIN  =   2.8     # calibrated omni antenna gain (dBi)
TX_CABLE_LOSS    =   2.8     # feeder/cable loss (dB)
TX_BODY_LOSS     =   0.0     # not specified

# RX drive-test parameters (Ofcom spec exact values)
RX_ANTENNA_GAIN  =  -2.0     # calibrated omni RX antenna (dBi) — negative = below isotropic
RX_CABLE_LOSS    =   3.8     # RX cable loss (dB)
RX_SPLITTER_LOSS =   0.0     # splitter loss (dB)
LNA_GAIN_DB      =  23.3     # LNA gain (dB)
RX_BPF_LOSS_DB   =   1.5     # band-pass filter loss (dB)
NOISE_FLOOR      = -109.0    # system noise floor (dBm)

# Verified totals from Ofcom spec
EIRP_DBM  =  54.0   # Ofcom verified: 47.8 - 2.8 + 2.8 + 6.2 correction = 54 dBm
SYS_GAIN  =  16.0   # Ofcom verified: -2.0 - 3.8 + 23.3 - 1.5 = 16.0 dB

# Legacy aliases (used elsewhere in notebook)
TX_POWER_DBM_EFF = EIRP_DBM
TX_GAIN_DBI      = TX_ANTENNA_GAIN
RX_GAIN_DBI      = RX_ANTENNA_GAIN
NOISE_FLOOR_DBM  = NOISE_FLOOR    # alias used in CELL 8 stats

NUM_SUBCARRIERS    = 76
SUBCARRIER_SPACING = 30e3
if globals().get('_HAS_OFDM') and globals().get('subcarrier_frequencies'):
    FREQUENCIES = subcarrier_frequencies(NUM_SUBCARRIERS, SUBCARRIER_SPACING)

MAX_DEPTH      = 6
NUM_SAMPLES_CM = 100_000_000 # rays fired for coverage map (~250 rays/cell at 20m grid, ~6 min on CPU)
NUM_SAMPLES_PS = 10_000_000 # rays fired for path solver (10M for full-scene coverage)
GRID_SIZE_M    = 20.0       # coverage map cell size (m) — increase if OOM

# GPU memory controls
# Set True to force CPU (llvm_ad_rgb) for ray tracing — slower but no OOM
FORCE_CPU_RT      = True   # GPU OOMs on 81k-building scene; CPU is safe
# Limit PLY scene to buildings within this radius of scene center (km)
# Reduces BVH size. None = full scene. Recommended: 5.0 for 8GB GPU.
SCENE_RADIUS_KM   = None

# Height above ground level
TX_AGL_M       = 17.0   # TX antenna height AGL (m) — Ofcom spec
RX_AGL_M       =  1.5   # RX height AGL (m) — Ofcom spec (vehicle roof)

_tx_w    = 10**((TX_POWER_DBM - 30) / 10)
_noise_w = 10**((NOISE_FLOOR  - 30) / 10)
SNR_SCALE = _tx_w / _noise_w

# ── Scene area size (sanity check) ────────────────────────────────────────────
_gps_to_utm_tmp = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
_sw = _gps_to_utm_tmp.transform(WEST,  SOUTH)
_ne = _gps_to_utm_tmp.transform(EAST,  NORTH)
_w_km = (_ne[0] - _sw[0]) / 1000
_h_km = (_ne[1] - _sw[1]) / 1000

print('=' * 60)
print(f'CITY          : {CITY_NAME}')
print(f'Scene bbox    : lon [{WEST:.6f}, {EAST:.6f}]')
print(f'                lat [{SOUTH:.6f}, {NORTH:.6f}]')
print(f'Area          : {_w_km:.2f} km x {_h_km:.2f} km')
print(f'Center        : ({center_lon:.6f}, {center_lat:.6f})')
print(f'UTM EPSG      : {UTM_EPSG}')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'TX power      : {TX_POWER_DBM:.1f} dBm  +  {TX_ANTENNA_GAIN:.1f} dBi  -  {TX_CABLE_LOSS:.1f} dB loss')
print(f'EIRP          : {EIRP_DBM:.1f} dBm')
print(f'RX gain       : {RX_ANTENNA_GAIN:.1f} dBi  |  System gain: {SYS_GAIN:.1f} dB')
print(f'DEM           : {DEM_TIFF}')
print(f'Output dir    : {OUT_DIR}')
print('=' * 60)

if _w_km > 10 or _h_km > 10:
    print(f'NOTE: Large area ({_w_km:.1f}x{_h_km:.1f} km). OSM download may take ~1 hour.')


## CELL 1 · Coordinate Utilities + DEM Elevation

- `gps_to_local(lon, lat)` → UTM → subtract scene origin → local XY
- `local_to_gps(x, y)` → reverse
- `get_dem_elevation(local_x, local_y)` → rasterio bilinear lookup
- `ray_cast_ground_z(x, y)` → Mitsuba ray intersect for terrain height

In [ ]:
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)
utm_to_bng = Transformer.from_crs(f'EPSG:{UTM_EPSG}', f'EPSG:{BNG_EPSG}', always_xy=True)

utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)
print(f'UTM center : ({utm_center_x:.1f}, {utm_center_y:.1f})')

def gps_to_local(lon, lat, height=0.0):
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    lon, lat = utm_to_gps.transform(_safe(x) + utm_center_x, _safe(y) + utm_center_y)
    return float(lon), float(lat)

dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM     : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    print('DEM     : not loaded (rasterio missing or file absent)')

_is_bng_dem = dem_crs is not None and ('27700' in dem_crs or 'OSGB' in dem_crs.upper())

def get_dem_elevation(local_x, local_y):
    if dem_data is None: return 0.0
    utm_x = _safe(local_x) + utm_center_x
    utm_y = _safe(local_y) + utm_center_y
    px, py = (utm_to_bng.transform(utm_x, utm_y) if _is_bng_dem
              else utm_to_gps.transform(utm_x, utm_y))
    col_f, row_f = ~dem_tf * (px, py)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

def ray_cast_ground_z(x, y, max_height=2000.0):
    if _HAS_MI:
        try:
            ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                           mi.Vector3f(0.0, 0.0, -1.0))
            si = scene.mi_scene.ray_intersect(ray)
            if si.is_valid():
                z_val = si.p.z
                return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
        except Exception: pass
    return get_dem_elevation(x, y)

print('Coordinate utilities ready.')
print(f'  gps_to_local({center_lon:.4f}, {center_lat:.4f}) → {gps_to_local(center_lon, center_lat)[:2]}')

## CELL 2 · OSM Map Download

Downloads **all buildings** from OpenStreetMap for the full scene bbox defined in CELL 0c.

- Full Nottingham (~16 km x 9 km, ~93k buildings): download takes ~45–90 min; cached to GeoJSON after first run.
- On re-run, loads instantly from the cached GeoJSON file.
- Buildings with no height tag default to **10 m**; tagged floors use **3 m/level**.


In [ ]:
if FORCE_REBUILD_SCENE or not os.path.exists(
        os.path.join(OUT_DIR, 'osm_buildings.geojson')):
    pass  # run block below
# ── OSM download (skip if scene already built) ────────────────────────────────
# Set FORCE_REBUILD_SCENE=True in CELL 0c to re-download

import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ── Paths ─────────────────────────────────────────────────────────────────────
OSM_GEOJSON = os.path.join(OUT_DIR, 'osm_buildings.geojson')
OSM_XML     = os.path.join(SCENE_DIR, 'scene', 'scene.xml')
os.makedirs(os.path.dirname(OSM_XML), exist_ok=True)

print(f'OSM download area : lon [{WEST:.5f}, {EAST:.5f}]  lat [{SOUTH:.5f}, {NORTH:.5f}]')
print(f'                  : {_w_km:.2f} km × {_h_km:.2f} km')

if not _HAS_OSM:
    print('\nosmnx not available. Install with:\n  pip install osmnx shapely')
else:
    import geopandas as gpd

    # ── Download or load from cache ───────────────────────────────────────────
    if os.path.exists(OSM_GEOJSON):
        print(f'\nLoading cached buildings from {OSM_GEOJSON} ...')
        gdf = gpd.read_file(OSM_GEOJSON)
        print(f'  Loaded {len(gdf)} buildings from cache.')
    else:
        print('\nDownloading buildings from OpenStreetMap ...')
        t0 = time.time()

        # osmnx 2.0.x API change:
        #   OLD (1.x): ox.features_from_bbox(north=N, south=S, east=E, west=W, tags=tags)
        #   NEW (2.0): ox.features_from_bbox(bbox=(west, south, east, north), tags=tags)
        try:
            ox.settings.log_level = 30   # WARNING only — suppress INFO spam (osmnx 2.0)
        except AttributeError:
            try: ox.settings.log_console = False   # fallback for osmnx 1.x
            except: pass
        ox.settings.timeout = 180

        gdf = ox.features_from_bbox(
            bbox=(WEST, SOUTH, EAST, NORTH),   # osmnx 2.0: (left, bottom, right, top)
            tags={'building': True}
        )
        print(f'  Downloaded {len(gdf)} features in {time.time()-t0:.1f}s')

        # Filter: keep only polygon footprints ≥ 20 m²
        gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
        gdf = gdf.to_crs('EPSG:4326')
        gdf_utm = gdf.to_crs(f'EPSG:{UTM_EPSG}')
        gdf     = gdf[gdf_utm.geometry.area >= 20.0].reset_index(drop=True)
        print(f'  After polygon + area filter: {len(gdf)} buildings')

        # Cache to GeoJSON: save geometry + useful OSM height/type tags
        _keep_cols = ['geometry']
        for _c in ['height','building:height','building:levels',
                   'building','building:material']:
            if _c in gdf.columns:
                # convert list/complex cells to string to avoid GeoJSON errors
                gdf[_c] = gdf[_c].apply(
                    lambda v: str(v) if isinstance(v, (list, dict)) else v)
                _keep_cols.append(_c)
        gdf[_keep_cols].to_file(OSM_GEOJSON, driver='GeoJSON')
        print(f'  Saved to {OSM_GEOJSON}  (cols: {_keep_cols})')

    # ── Building height extraction ─────────────────────────────────────────────
    # Priority: OSM height tag > building:levels > SRTM nDSM > OSM type heuristic
    LEVEL_HEIGHT_M = 3.0

    # OSM building-type heuristic defaults (floors × 3 m)
    _TYPE_FLOORS = {
        'house': 2, 'detached': 2, 'semidetached_house': 2, 'terrace': 2,
        'bungalow': 1, 'static_caravan': 1, 'shed': 1, 'garage': 1, 'garages': 1,
        'residential': 3, 'apartments': 5, 'block_of_flats': 5,
        'commercial': 3, 'retail': 2, 'supermarket': 2, 'kiosk': 1,
        'office': 6, 'hotel': 7,
        'industrial': 2, 'warehouse': 1, 'factory': 2,
        'church': 6, 'cathedral': 8, 'chapel': 4, 'mosque': 4, 'temple': 3,
        'school': 3, 'university': 4, 'hospital': 5,
        'train_station': 4, 'transportation': 3,
        'stadium': 8, 'grandstand': 5,
        'yes': 3,   # generic tagged building
    }



    def _parse_height(row):
        # 1. Explicit OSM height tag
        for col in ('height', 'building:height'):
            if col in row and pd.notna(row[col]):
                try: return float(str(row[col]).split()[0])
                except: pass
        # 2. Building levels tag
        if 'building:levels' in row and pd.notna(row.get('building:levels')):
            try: return float(row['building:levels']) * LEVEL_HEIGHT_M
            except: pass
        # 3. OSM building-type heuristic
        btype = str(row.get('building', '')).strip().lower()
        if btype in _TYPE_FLOORS:
            return _TYPE_FLOORS[btype] * LEVEL_HEIGHT_M
        return 8.0   # final fallback

    heights = gdf.apply(_parse_height, axis=1).values
    n_osm    = int(sum(1 for _, r in gdf.iterrows()
                       for c in ('height','building:height')
                       if c in r and pd.notna(r[c])))
    n_levels = int(sum(1 for _, r in gdf.iterrows()
                       if 'building:levels' in r and pd.notna(r.get('building:levels'))))
    print(f'\n  Building heights: min={heights.min():.1f}  mean={heights.mean():.1f}  '
          f'max={heights.max():.1f} m')
    # Clip to city-configured height range (set CITY_MAX_HEIGHT_M in CELL 0c)
    heights = np.clip(heights, CITY_MIN_HEIGHT_M, CITY_MAX_HEIGHT_M)

    print(f'  Sources: OSM tag={n_osm}  levels={n_levels}  '
          f'nDSM+heuristic={len(gdf)-n_osm-n_levels}')
    print(f'  Height stats: min={heights.min():.1f}  mean={heights.mean():.1f}  '
          f'max={heights.max():.1f} m  |  cap [{CITY_MIN_HEIGHT_M}, {CITY_MAX_HEIGHT_M}] m')

    # ── Plot ───────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Map — colour by height
    gdf.plot(ax=axes[0], column=heights, cmap='YlOrRd',
             legend=True, legend_kwds={'label': 'Height (m)'},
             edgecolor='k', linewidth=0.2, alpha=0.8)
    axes[0].add_patch(Rectangle((WEST, SOUTH), EAST-WEST, NORTH-SOUTH,
                                 fill=False, edgecolor='blue', lw=2, label='OSM bbox'))
    axes[0].set_title(f'{CITY_NAME} – OSM Buildings ({len(gdf):,})')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    axes[0].legend()

    # Height histogram
    axes[1].hist(heights, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(8.0, color='red', ls='--', label='Final fallback (8 m)')
    axes[1].set_xlabel('Building Height (m)'); axes[1].set_ylabel('Count')
    axes[1].set_title('Building Height Distribution'); axes[1].legend()

    plt.suptitle(f'{CITY_NAME} OSM Download  |  {len(gdf):,} buildings', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'osm_buildings.png'), dpi=150)
    plt.show()
    print(f'Map saved → {os.path.join(OUT_DIR, "osm_buildings.png")}')

## CELL 3 · Generate Mitsuba 3 Scene XML from OSM Buildings

Converts OSM footprints + heights into a Sionna 0.19-compatible Mitsuba 3 scene XML.
Each building is extruded as a box mesh. Ground plane is added automatically.

> Skip this cell if you already have a `scene.xml` from sionna_web.

In [ ]:
# OSM -> Mitsuba 3 scene: PLY meshes + scene.xml (Sionna 0.19 compatible)
# Skipped automatically if scene.xml exists and FORCE_REBUILD_SCENE=False

_scene_xml_exists = os.path.exists(SCENE_XML)
if not FORCE_REBUILD_SCENE and _scene_xml_exists:
    print(f'Scene already exists: {SCENE_XML}')
    print('Set FORCE_REBUILD_SCENE=True in CELL 0c to regenerate.')
    XML_OK = True
else:
    # Buildings grouped by material -> one PLY per material type.
    # Terrain sampled from DEM -> terrain.ply
    # All PLY files in scene/meshes/; scene.xml references them by filename.

    import xml.etree.ElementTree as ET
    import xml.dom.minidom as minidom
    import numpy as np
    import struct, os, time

    _HAS_DEM_RASTERIO = False
    try:
        import rasterio
        _HAS_DEM_RASTERIO = True
    except ImportError:
        pass

    if not _HAS_OSM or 'gdf' not in dir():
        print('Run CELL 2 first.')
    else:
        print(f'Generating Mitsuba 3 scene (PLY + XML) from {len(gdf)} buildings ...')
        t0 = time.time()

        SCENE_OUT_DIR  = os.path.join(BASE_DIR, 'scene')
        MESHES_DIR     = os.path.join(SCENE_OUT_DIR, 'meshes')
        os.makedirs(MESHES_DIR, exist_ok=True)
        SCENE_XML_OUT  = os.path.join(SCENE_OUT_DIR, 'scene.xml')

        # ── Load DEM ──────────────────────────────────────────────────────────
        _dem_ds2 = _dem_data2 = _dem_tf2 = _wgs_to_dem2 = None
        if _HAS_DEM_RASTERIO and os.path.exists(DEM_TIFF):
            try:
                from pyproj import Transformer as _Tr2
                _dem_ds2   = rasterio.open(DEM_TIFF)
                _dem_data2 = _dem_ds2.read(1).astype(np.float32)
                _dem_tf2   = _dem_ds2.transform
                _epsg2     = str(_dem_ds2.crs.to_epsg()) if _dem_ds2.crs else ''
                _wgs_to_dem2 = _Tr2.from_crs('EPSG:4326',
                                   f'EPSG:{_epsg2}' if _epsg2 else _dem_ds2.crs,
                                   always_xy=True)
                print(f'  DEM: {_dem_data2.shape} px, EPSG:{_epsg2}')
            except Exception as _e:
                print(f'  DEM load failed: {_e}')

        def _dem_z2(lon, lat):
            if _dem_data2 is None: return 0.0
            try:
                dx, dy = _wgs_to_dem2.transform(lon, lat)
                tf = _dem_tf2
                col = (dx - tf.c) / tf.a
                row = (dy - tf.f) / tf.e
                r, c = int(round(row)), int(round(col))
                nr, nc = _dem_data2.shape
                r = max(0, min(nr-1, r)); c = max(0, min(nc-1, c))
                v = float(_dem_data2[r, c])
                return v if (np.isfinite(v) and v > -9000) else 0.0
            except Exception: return 0.0

        origin_elev2 = _dem_z2(center_lon, center_lat)

        # ── PLY writer ────────────────────────────────────────────────────────
        def write_ply(path, verts, faces):
            verts = np.asarray(verts, dtype=np.float32)
            faces = np.asarray(faces, dtype=np.int32)
            with open(path, 'wb') as fp:
                hdr = (
                    'ply\nformat binary_little_endian 1.0\n'
                    f'element vertex {len(verts)}\n'
                    'property float x\nproperty float y\nproperty float z\n'
                    f'element face {len(faces)}\n'
                    'property list uchar int vertex_indices\n'
                    'end_header\n'
                )
                fp.write(hdr.encode())
                fp.write(verts.tobytes())
                for tri in faces:
                    fp.write(struct.pack('<B', 3))
                    fp.write(struct.pack('<3i', *tri))

        # ── Project OSM to UTM ────────────────────────────────────────────────
        gdf_utm = gdf.to_crs(f'EPSG:{UTM_EPSG}')
        ox_utm, oy_utm = utm_center_x, utm_center_y

        # Optional radius filter — reduces BVH size / GPU memory
        if SCENE_RADIUS_KM is not None:
            _r = SCENE_RADIUS_KM * 1000.0
            _cx = gdf_utm.geometry.centroid.x - ox_utm
            _cy = gdf_utm.geometry.centroid.y - oy_utm
            _mask = (_cx**2 + _cy**2) <= _r**2
            _before = len(gdf_utm)
            gdf_utm = gdf_utm[_mask].reset_index(drop=True)
            gdf     = gdf[_mask].reset_index(drop=True)
            print(f'  Radius filter {SCENE_RADIUS_KM} km: {_before} -> {len(gdf_utm)} buildings')

        ITU_MATS       = ['itu_concrete','itu_brick','itu_glass','itu_wood']
        ITU_ROOF_MATS  = ['itu_metal']  # gabled/hip roof faces

        def _bld_mat(row):
            mat = str(row.get('building:material','')).lower()
            tag = str(row.get('building','')).lower()
            if 'glass' in mat:                      return 'itu_glass'
            if 'wood' in mat or 'timber' in mat:    return 'itu_wood'
            if 'brick' in mat:                      return 'itu_brick'
            if tag in ('greenhouse','glasshouse'):  return 'itu_glass'
            return 'itu_concrete'

        # Collect per-material vertex/face lists
        mat_verts = {m: [] for m in ITU_MATS + ITU_ROOF_MATS}
        mat_faces = {m: [] for m in ITU_MATS + ITU_ROOF_MATS}

        gdf_wgs_list  = list(gdf.iterrows())
        gdf_utm_list  = list(gdf_utm.iterrows())
        skipped = 0

        for idx in range(len(gdf_utm_list)):
            _, row_utm = gdf_utm_list[idx]
            _, row_wgs = gdf_wgs_list[idx]
            geom = row_utm.geometry
            if geom is None or geom.is_empty:
                skipped += 1; continue
            try:
                hull = geom.convex_hull
                if hull.geom_type != 'Polygon': skipped += 1; continue
                ring = list(hull.exterior.coords[:-1])
            except Exception:
                skipped += 1; continue
            n = len(ring)
            if n < 3: skipped += 1; continue

            # building height
            h = heights[idx] if idx < len(heights) else 8.0

            # base elevation from DEM
            try:
                ctr = row_wgs.geometry.centroid
                base_z = _dem_z2(ctr.x, ctr.y) - origin_elev2
            except Exception:
                base_z = 0.0

            pts = [(x - ox_utm, y - oy_utm) for x, y in ring]
            z0, z1 = base_z, base_z + h

            mat = _bld_mat(dict(row_wgs))
            vl  = mat_verts[mat]
            fl  = mat_faces[mat]
            base_idx = len(vl)

            # bottom ring + top ring (wall vertices)
            for x, y in pts: vl.append((x, y, z0))
            for x, y in pts: vl.append((x, y, z1))

            # side walls
            for i in range(n):
                j = (i+1) % n
                b, t = base_idx, base_idx + n
                fl.append((b+i, b+j, t+j))
                fl.append((b+i, t+j, t+i))
            # bottom cap fan (reversed)
            for i in range(1, n-1):
                fl.append((base_idx, base_idx+i+1, base_idx+i))

            # ── Hip (pyramid) metal roof ──────────────────────────────────────
            # Ridge apex at centroid + ROOF_HEIGHT above wall top
            ROOF_HEIGHT = max(1.5, h * 0.25)   # 25% of wall height, min 1.5m
            cx = sum(x for x,y in pts) / n
            cy = sum(y for x,y in pts) / n
            rvl = mat_verts['itu_metal']
            rfl = mat_faces['itu_metal']
            roof_base = len(rvl)
            # eave ring (same as wall top ring)
            for x, y in pts:
                rvl.append((x, y, z1))
            # apex
            rvl.append((cx, cy, z1 + ROOF_HEIGHT))
            apex_idx = roof_base + n
            for i in range(n):
                j = (i+1) % n
                rfl.append((roof_base+i, roof_base+j, apex_idx))

        print(f'  Built geometry: {sum(len(v) for v in mat_verts.values()):,} verts'
              f'  {sum(len(f) for f in mat_faces.values()):,} faces  ({skipped} skipped)')

        # ── Write building PLY files (walls + metal roofs) ─────────────────────
        ply_files = {}
        for mat in ITU_MATS + ITU_ROOF_MATS:
            if not mat_verts[mat]: continue
            ply_path = os.path.join(MESHES_DIR, f'{mat}.ply')
            write_ply(ply_path, mat_verts[mat], mat_faces[mat])
            ply_files[mat] = ply_path
            print(f'  {mat}.ply  {len(mat_verts[mat]):,} verts  {len(mat_faces[mat]):,} faces')

        # ── Terrain PLY (DEM 64x64 grid) ──────────────────────────────────────
        DEM_GRID_N = 64
        lons_g = np.linspace(WEST,  EAST,  DEM_GRID_N)
        lats_g = np.linspace(SOUTH, NORTH, DEM_GRID_N)
        t_verts, t_faces = [], []
        for lat in lats_g:
            for lon in lons_g:
                ux, uy = gps_to_utm.transform(lon, lat)
                lx, ly = ux - ox_utm, uy - oy_utm
                z = _dem_z2(lon, lat) - origin_elev2
                t_verts.append((lx, ly, z))
        for r in range(DEM_GRID_N-1):
            for c in range(DEM_GRID_N-1):
                i00 = r*DEM_GRID_N + c
                i10 = i00 + 1
                i01 = i00 + DEM_GRID_N
                i11 = i01 + 1
                t_faces += [(i00,i10,i11), (i00,i11,i01)]
        terrain_ply = os.path.join(MESHES_DIR, 'terrain.ply')
        write_ply(terrain_ply, t_verts, t_faces)
        print(f'  terrain.ply  {len(t_verts)} verts  {len(t_faces)} faces')

        # ── scene.xml ─────────────────────────────────────────────────────────
        root = ET.Element('scene', version='3.0.0')
        for name, val in [
            ('scenegen_min_lon',    str(WEST)),
            ('scenegen_max_lon',    str(EAST)),
            ('scenegen_min_lat',    str(SOUTH)),
            ('scenegen_max_lat',    str(NORTH)),
            ('scenegen_origin_lon', str(center_lon)),
            ('scenegen_origin_lat', str(center_lat)),
        ]:
            ET.SubElement(root, 'default', name=name, value=val)

        integ = ET.SubElement(root, 'integrator', type='path')
        ET.SubElement(integ, 'integer', name='max_depth', value='8')

        # ── scene.xml: inline BSDF inside each shape (required by Sionna 0.19)
        # Global <bsdf> + <ref id="..."> does NOT work — shapes are dropped silently.
        # Each shape gets its own inline <bsdf type="diffuse" id="itu_*"> so Sionna
        # can identify radio materials from the itu_ prefix after load_scene.

        # terrain shape
        s = ET.SubElement(root, 'shape', type='ply', id='terrain')
        ET.SubElement(s, 'string', name='filename',
                      value=os.path.relpath(terrain_ply, SCENE_OUT_DIR))
        bsdf = ET.SubElement(s, 'bsdf', type='diffuse', id='itu_wet_ground')
        ET.SubElement(bsdf, 'rgb', name='reflectance', value='0.12 0.22 0.28')  # dark blue-grey: wet ground

        # building shapes — inline BSDF per material
        # Visually distinct colors — buildings contrast against dark wet-ground floor
        _MAT_REFLECTANCE = {
            'itu_concrete' : '0.82 0.82 0.82',   # bright grey
            'itu_brick'    : '0.85 0.28 0.15',   # vivid red-brown
            'itu_glass'    : '0.25 0.75 0.95',   # bright cyan-blue
            'itu_wood'     : '0.70 0.45 0.12',   # warm brown
            'itu_metal'    : '0.90 0.90 0.92',   # near-silver (galvanised steel)
        }
        for mat, ply_path in ply_files.items():
            shape_id = mat.replace('itu_', '') + ('_roofs' if mat == 'itu_metal' else '_buildings')
            s = ET.SubElement(root, 'shape', type='ply', id=shape_id)
            ET.SubElement(s, 'string', name='filename',
                          value=os.path.relpath(ply_path, SCENE_OUT_DIR))
            refl = _MAT_REFLECTANCE.get(mat, '0.5 0.5 0.5')
            bsdf = ET.SubElement(s, 'bsdf', type='diffuse', id=mat)
            ET.SubElement(bsdf, 'rgb', name='reflectance', value=refl)

        xml_str = minidom.parseString(
            ET.tostring(root, encoding='unicode')
        ).toprettyxml(indent='  ', encoding=None)
        with open(SCENE_XML_OUT, 'w', encoding='utf-8') as f:
            f.write(xml_str)

        SCENE_XML = SCENE_XML_OUT
        XML_OK    = True
        if _dem_ds2: _dem_ds2.close()

        print(f'\n  Done in {time.time()-t0:.1f}s')
        print(f'  scene/meshes/ -> {len(ply_files)+1} PLY files')
        print(f'  scene.xml     -> {SCENE_XML_OUT}')
        print('Ready for CELL 4: load_scene(SCENE_XML)')


# Redefine ray_cast_ground_z with live scene (CELL 1 ran before scene was loaded)
def ray_cast_ground_z(x, y, max_height=2000.0):
    try:
        ray = mi.Ray3f(mi.Point3f(float(x), float(y), max_height),
                       mi.Vector3f(0.0, 0.0, -1.0))
        si = scene.mi_scene.ray_intersect(ray)
        if si.is_valid():
            z_val = si.p.z
            return float(z_val.item()) if hasattr(z_val, 'item') else float(z_val)
    except Exception:
        pass
    return get_dem_elevation(x, y)

print('ray_cast_ground_z redefined with live scene.')

## CELL 4 · Load 3-D Scene & Configure Antennas

**Sionna 0.19 API:** `load_scene(path)` — no `merge_shapes` argument.

In [ ]:
XML_OK = os.path.exists(SCENE_XML)
if not XML_OK:
    raise RuntimeError(
        f'scene.xml not found at {SCENE_XML}\n'
        'Either:\n'
        '  A) Run CELL 3 to generate from OSM (requires Blender for full mesh)\n'
        '  B) Generate via sionna_web Steps 1-3\n'
        '  C) Use Blender + BlenderOSM plugin')

# Variant was set in CELL 0 before sionna import — do not change it here.
print(f'Mitsuba variant : {mi.variant()}')
print(f'Loading scene from {SCENE_XML} ...')
scene = load_scene(SCENE_XML)   # Sionna 0.19: NO merge_shapes argument
scene.frequency = FREQUENCY_HZ

scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')

print(f'Scene loaded  : {len(scene.objects)} objects,  {len(scene.radio_materials)} materials')
print(f'Frequency     : {FREQUENCY_HZ/1e9:.3f} GHz')
print(f'Materials     : {list(scene.radio_materials.keys())}')

In [ ]:
# ====================================================================
# CELL 4b — SCENE PREVIEW  [Sionna 0.19]
# ====================================================================
# Requires: conda install -c conda-forge pythreejs ipywidgets
#           jupyter nbextension enable --py widgetsnbextension
#           jupyter nbextension enable --py pythreejs
# Restart the kernel after installing.
# ====================================================================

try:
    import pythreejs  # noqa
    _HAS_PYTHREEJS = True
except ImportError:
    _HAS_PYTHREEJS = False
    print('pythreejs not installed.')
    print('Run in terminal (sionna019 env):')
    print('  conda install -c conda-forge pythreejs ipywidgets -y')
    print('  jupyter nbextension enable --py widgetsnbextension')
    print('  jupyter nbextension enable --py pythreejs')
    print('Then restart kernel and re-run this cell.')

if _HAS_PYTHREEJS:
    print('Launching scene preview ...')
    print('(rotate: left-drag | zoom: scroll | pan: right-drag)')
    # Show scene geometry only (no paths)
    scene.preview()


In [ ]:
# ====================================================================
# CELL 4c — SCENE DIAGNOSTICS (run before CELL 8 if RSSI = -156 dBm)
# ====================================================================
import numpy as np, os, mitsuba as mi

print('=' * 70)
print('SCENE DIAGNOSTICS')
print('=' * 70)

# ── 1. PLY files ──────────────────────────────────────────────────────────
meshes_dir = os.path.join(BASE_DIR, 'scene', 'meshes')
print('\n[1] PLY files:')
if os.path.isdir(meshes_dir):
    for fn in sorted(os.listdir(meshes_dir)):
        fp = os.path.join(meshes_dir, fn)
        sz = os.path.getsize(fp) / 1024
        print(f'  {fn:<28} {sz:8.1f} KB')
else:
    print(f'  ⚠ meshes dir not found: {meshes_dir}')

# ── 2. Scene bounding box ─────────────────────────────────────────────────
print('\n[2] Scene bbox (local coords, metres):')
try:
    bbox = scene.mi_scene.bbox()
    print(f'  X: [{float(bbox.min[0]):.1f}, {float(bbox.max[0]):.1f}]  '
          f'W={float(bbox.max[0]-bbox.min[0]):.0f} m')
    print(f'  Y: [{float(bbox.min[1]):.1f}, {float(bbox.max[1]):.1f}]  '
          f'H={float(bbox.max[1]-bbox.min[1]):.0f} m')
    print(f'  Z: [{float(bbox.min[2]):.1f}, {float(bbox.max[2]):.1f}]  '
          f'depth={float(bbox.max[2]-bbox.min[2]):.0f} m')
except Exception as e:
    print(f'  ⚠ bbox error: {e}')

# ── 3. Transmitter position ───────────────────────────────────────────────
print('\n[3] Transmitters:')
for nm, t in scene.transmitters.items():
    px = _safe(t.position[0]); py = _safe(t.position[1]); pz = _safe(t.position[2])
    lon, lat = local_to_gps(px, py)
    print(f'  "{nm}"  local=({px:.1f}, {py:.1f}, {pz:.2f}) m  GPS=({lon:.5f},{lat:.5f})')
    print(f'  EIRP used in formula = {EIRP_DBM:.1f} dBm')

# ── 4. Ray casts from TX ──────────────────────────────────────────────────
print('\n[4] Ray casts from TX:')
try:
    if not scene.transmitters:
        _tx_x, _tx_y = gps_to_local(TX_LON, TX_LAT)
        _tx_z = get_dem_elevation(_tx_x, _tx_y) + TX_AGL_M
        print(f'  (No TX loaded — using config GPS as synthetic position: ({_tx_x:.1f}, {_tx_y:.1f}, {_tx_z:.2f}) m)')
    else:
        _tx   = list(scene.transmitters.values())[0]
        _tx_x = _safe(_tx.position[0])
        _tx_y = _safe(_tx.position[1])
        _tx_z = _safe(_tx.position[2])

    # Cast down (should hit terrain/building below)
    for label, direction in [
        ('downward  (-Z)', mi.Vector3f(0, 0, -1)),
        ('upward    (+Z)', mi.Vector3f(0, 0,  1)),
        ('horizontal +X', mi.Vector3f(1, 0,  0)),
        ('horizontal +Y', mi.Vector3f(0, 1,  0)),
    ]:
        try:
            ray = mi.Ray3f(mi.Point3f(_tx_x, _tx_y, _tx_z), direction)
            si  = scene.mi_scene.ray_intersect(ray)
            if si.is_valid():
                hit_p = si.p
                dist  = float(si.t)
                print(f'  {label}: HIT at dist={dist:.1f} m  z={float(hit_p.z):.2f} m')
            else:
                print(f'  {label}: NO HIT (ray escapes scene)')
        except Exception as e:
            print(f'  {label}: ERROR {e}')
except Exception as e:
    print(f'  ⚠ ray cast error: {e}')

# ── 5. First receiver ─────────────────────────────────────────────────────
print('\n[5] First 3 receivers:')
for rx in receivers[:3]:
    px = _safe(rx.position[0]); py = _safe(rx.position[1]); pz = _safe(rx.position[2])
    lon, lat = local_to_gps(px, py)
    print(f'  "{rx.name}"  local=({px:.1f},{py:.1f},{pz:.2f})  GPS=({lon:.5f},{lat:.5f})')

# ── 6. Mini coverage map test (100x100 m around TX, many samples) ─────────
print('\n[6] Mini coverage map (200x200 m around TX, 1M samples):')
try:
    if not scene.transmitters:
        _tx_x2, _tx_y2 = gps_to_local(TX_LON, TX_LAT)
        _cx = _tx_x2; _cy = _tx_y2
        _cz = get_dem_elevation(_tx_x2, _tx_y2) + RX_AGL_M
        print('  (No TX loaded — using config GPS; run CELL 6 first for accurate results)')
    else:
        _tx   = list(scene.transmitters.values())[0]
        _cx   = _safe(_tx.position[0]); _cy = _safe(_tx.position[1])
        _cz   = _safe(_tx.position[2]) - TX_AGL_M + RX_AGL_M  # RX height

    _cm_mini = scene.coverage_map(
        cm_center      = (_cx, _cy, _cz),
        cm_orientation = (0., 0., 0.),
        cm_size        = (200., 200.),
        cm_cell_size   = (5., 5.),
        max_depth      = 3,
        num_samples    = 1_000_000,
        los=True, reflection=True, scattering=False, diffraction=False)

    _pg = np.array(_cm_mini.path_gain)
    print(f'  path_gain shape : {_pg.shape}')
    print(f'  path_gain range : min={_pg.min():.2e}  max={_pg.max():.2e}')
    _pg2d = _pg[0] if _pg.ndim == 3 else _pg
    _nz   = (_pg2d > 1e-20).sum()
    print(f'  Non-zero cells  : {_nz} / {_pg2d.size}  ({100*_nz/max(_pg2d.size,1):.1f}%)')
    if _nz > 0:
        _pl = -10 * np.log10(np.maximum(_pg2d[_pg2d > 1e-20], 1e-20))
        _rs = EIRP_DBM - _pl + RX_GAIN_DBI + LNA_GAIN_DB
        print(f'  RSSI (non-zero) : mean={_rs.mean():.1f}  min={_rs.min():.1f}  max={_rs.max():.1f} dBm')
        print(f'  ✅ Coverage map is working — scale up NUM_SAMPLES_CM for full scene')
    else:
        print(f'  ⚠ ALL cells zero — scene/TX geometry issue (see checks above)')
except Exception as e:
    print(f'  ⚠ mini coverage map error: {e}')

# ── 7. Scene objects count ───────────────────────────────────────────────────
print('\n[7] Scene objects:')
try:
    print(f'  scene.objects      : {len(scene.objects)}')
    print(f'  scene.radio_materials: {list(scene.radio_materials.keys())}')
    for obj_name, obj in list(scene.objects.items())[:5]:
        print(f'  obj "{obj_name}"')
except Exception as e:
    print(f'  ⚠ {e}')

# ── 8. Single close-RX path test ─────────────────────────────────────────────
print('\n[8] Single close-range path test (RX 50 m from TX):')
try:
    import drjit as dr
    if not scene.transmitters:
        _tx_xp, _tx_yp = gps_to_local(TX_LON, TX_LAT)
        _txp = [_tx_xp, _tx_yp, get_dem_elevation(_tx_xp, _tx_yp) + TX_AGL_M]
        print('  (No TX loaded — using config GPS; run CELL 6 first)')
    else:
        _tx  = list(scene.transmitters.values())[0]
        _txp = [_safe(_tx.position[i]) for i in range(3)]
    _test_rx = Receiver(name='_diag_rx',
                        position=(_txp[0]+50., _txp[1], _txp[2]-TX_AGL_M+RX_AGL_M))
    for _nm in list(scene.receivers.keys()):
        scene.remove(_nm)
    scene.add(_test_rx)
    _paths = scene.compute_paths(
        max_depth=3, num_samples=500_000,
        los=True, reflection=True, scattering=False, diffraction=False)
    _a = np.array(_paths.a)
    print(f'  paths.a shape : {_a.shape}')
    print(f'  paths.a max   : {np.abs(_a).max():.3e}')
    _tot = (np.abs(_a)**2).sum()
    if _tot > 1e-30:
        _pl = -10*np.log10(float(_tot))
        print(f'  Path loss (incoherent): {_pl:.1f} dB')
        print(f'  RSSI: {EIRP_DBM - _pl + RX_GAIN_DBI:.1f} dBm')
        print('  ✅ Paths found — scene geometry OK')
    else:
        print('  ⚠ Zero path gain — TX/RX geometry or scene issue')
    scene.remove('_diag_rx')
except Exception as e:
    print(f'  ⚠ {e}')
finally:
    # Restore all receivers
    try:
        for _nm in list(scene.receivers.keys()):
            scene.remove(_nm)
        for _rx in receivers:
            scene.add(_rx)
    except:
        pass

print('\n' + '=' * 70)


In [ ]:
# ====================================================================
# CELL 4d — SCENE.XML INSPECTOR + STANDALONE PLY TEST
# ====================================================================
import os, struct, numpy as np

# ── 1. Read scene.xml and count shapes ───────────────────────────────────────
print('[1] scene.xml content check:')
print(f'  Path: {SCENE_XML}')
print(f'  Exists: {os.path.exists(SCENE_XML)}')
if os.path.exists(SCENE_XML):
    with open(SCENE_XML) as f:
        xml_lines = f.readlines()
    print(f'  Lines: {len(xml_lines)}')
    shapes   = [l.strip() for l in xml_lines if '<shape' in l]
    bsdfs    = [l.strip() for l in xml_lines if '<bsdf'  in l]
    refs     = [l.strip() for l in xml_lines if '<ref'   in l]
    filenames= [l.strip() for l in xml_lines if 'filename' in l]
    print(f'  <shape> tags : {len(shapes)}')
    print(f'  <bsdf>  tags : {len(bsdfs)}')
    print(f'  <ref>   tags : {len(refs)}')
    print(f'  filename refs: {len(filenames)}')
    print('  First 30 lines:')
    for l in xml_lines[:30]:
        print('   ', l.rstrip())

# ── 2. scene.objects after load ───────────────────────────────────────────────
print('\n[2] scene.objects:')
print(f'  len(scene.objects)         = {len(scene.objects)}')
print(f'  len(scene.radio_materials) = {len(scene.radio_materials)}')
if scene.objects:
    for name, obj in list(scene.objects.items())[:10]:
        print(f'  obj: {name}')
else:
    print('  ⚠ ZERO objects — PLY files not loaded by Sionna')

# ── 3. Read first 3 vertices of terrain.ply directly ─────────────────────────
print('\n[3] terrain.ply raw vertex check:')
ply_path = os.path.join(BASE_DIR, 'scene', 'meshes', 'terrain.ply')
if os.path.exists(ply_path):
    with open(ply_path, 'rb') as f:
        # Read header
        header = b''
        while b'end_header' not in header:
            header += f.read(1)
        hdr_str = header.decode('ascii', errors='ignore')
        # Count vertices from header
        n_verts = 0
        for line in hdr_str.split('\n'):
            if 'element vertex' in line:
                n_verts = int(line.split()[-1])
        print(f'  Vertex count in header: {n_verts}')
        # Read first 3 vertices (12 bytes each = 3 floats)
        raw = f.read(36)
        verts = struct.unpack('<9f', raw)
        print(f'  First 3 vertices (x,y,z):')
        for i in range(3):
            print(f'    v{i}: ({verts[i*3]:.2f}, {verts[i*3+1]:.2f}, {verts[i*3+2]:.2f})')
else:
    print(f'  ⚠ Not found: {ply_path}')

# ── 4. Read first 3 vertices of concrete.ply ─────────────────────────────────
print('\n[4] itu_concrete.ply vertex range:')
ply_path2 = os.path.join(BASE_DIR, 'scene', 'meshes', 'itu_concrete.ply')
if os.path.exists(ply_path2):
    with open(ply_path2, 'rb') as f:
        header = b''
        while b'end_header' not in header:
            header += f.read(1)
        hdr_str = header.decode('ascii', errors='ignore')
        n_verts = 0
        for line in hdr_str.split('\n'):
            if 'element vertex' in line:
                n_verts = int(line.split()[-1])
        print(f'  Vertex count: {n_verts:,}')
        # Sample first, middle, last 3 vertices
        raw = f.read(n_verts * 12)
        arr = np.frombuffer(raw, dtype=np.float32).reshape(-1, 3)
        print(f'  X range: [{arr[:,0].min():.1f}, {arr[:,0].max():.1f}] m')
        print(f'  Y range: [{arr[:,1].min():.1f}, {arr[:,1].max():.1f}] m')
        print(f'  Z range: [{arr[:,2].min():.1f}, {arr[:,2].max():.1f}] m')
        print(f'  First vertex: ({arr[0,0]:.2f}, {arr[0,1]:.2f}, {arr[0,2]:.2f})')
        print(f'  TX position : see [DIAG] in CELL 8 output')
else:
    print(f'  ⚠ Not found: {ply_path2}')


## CELL 5 · Assign ITU-R P.2040-2 Material Properties

In [ ]:
# ====================================================================
# CELL 5 — ITU-R P.2040-2 Material Properties
# ====================================================================
# Sets: scattering_coefficient, xpd_coefficient, thickness per material.
# relative_permittivity / conductivity: Sionna 0.19 auto-computes from
# ITU-R tables at scene.frequency — only override if needed.
#
# HOW TO SEE THE EFFECT:
#   1. Edit the values in _ITU_DB below (S, xpd, thick in particular).
#   2. Re-run this cell.
#   3. Re-run CELL 8 (coverage map) — the new scattering/penetration
#      parameters will be used automatically.
#
# ENABLE_GRAD: set True to enable DrJit gradients on S, xpd, thickness
# for differentiable optimisation (calibration notebook).
# ====================================================================
import drjit as dr

ENABLE_GRAD = False   # True = enable dr.enable_grad() on S, xpd, thickness

# ── Parameter table ─────────────────────────────────────────────────────────
# (eps_r, sigma [S/m], scattering_coeff S [0-1], xpd_coeff [0-1], thickness [m])
#
#  eps_r   : relative permittivity  → reflection amplitude
#  sigma   : conductivity [S/m]     → absorption
#  S       : scattering coefficient → 0=specular only, 1=fully diffuse
#  xpd     : cross-polarisation     → depolarisation on scatter
#  thick   : wall thickness [m]     → penetration loss through walls
#
# Tweak S (scattering) and thick (thickness) to see the biggest effect
# on RSSI coverage maps. High S → more diffuse scatter, lower peaks.
# ────────────────────────────────────────────────────────────────────────────
_ITU_DB = {
    # material           eps_r   sigma    S     xpd   thick(m)
    'concrete'          : (5.24,  0.130, 0.40, 0.20, 0.20),
    'brick'             : (3.91,  0.024, 0.30, 0.20, 0.12),
    'glass'             : (6.27,  0.012, 0.08, 0.10, 0.01),
    'wood'              : (1.99,  0.005, 0.25, 0.30, 0.05),
    'metal'             : (1.00,  1e7,   0.05, 0.10, 0.01),
    'asphalt'           : (3.00,  0.010, 0.35, 0.20, 0.05),
    'vegetation'        : (1.30,  0.001, 0.75, 0.05, 0.10),
    'water'             : (81.0,  0.500, 0.02, 0.05, 0.01),
    'wet_ground'        : (30.0,  0.150, 0.20, 0.20, 0.50),
    'medium_dry_ground' : (15.0,  0.035, 0.18, 0.20, 0.50),
    'very_dry_ground'   : (3.00,  0.001, 0.12, 0.20, 0.50),
    'marble'            : (7.07,  0.020, 0.08, 0.10, 0.05),
    'plasterboard'      : (2.73,  0.010, 0.12, 0.20, 0.02),
    'plywood'           : (2.90,  0.013, 0.20, 0.25, 0.02),
    'chipboard'         : (2.58,  0.011, 0.22, 0.25, 0.02),
    'floorboard'        : (2.50,  0.008, 0.18, 0.25, 0.02),
    'ceiling_board'     : (2.25,  0.006, 0.15, 0.20, 0.02),
}
_DEFAULT_MAT = (4.0, 0.08, 0.30, 0.15, 0.10)

def _match_itu(mat_name):
    n = mat_name.lower().replace('itu_', '').replace('mat-', '').replace('mat_', '')
    for key in _ITU_DB:
        if key == n: return key
    for key in _ITU_DB:
        if key in n: return key
    for key in _ITU_DB:
        if any(part in n for part in key.split('_')): return key
    return None

print('=' * 78)
print('CELL 5 — ITU-R MATERIAL PROPERTIES')
print('=' * 78)
print(f'  Gradient enabled  : {ENABLE_GRAD}')
print(f'  Scene frequency   : {scene.frequency/1e6:.1f} MHz')
print(f'  Materials in scene: {len(scene.radio_materials)}')
print()
print(f'  {"Material":<28} {"eps_r":>6} {"sigma":>8} {"S":>5} {"xpd":>5} {"thick":>7}  Notes')
print('-' * 78)

for mat_name, mat in scene.radio_materials.items():
    if mat_name == 'vacuum':
        continue   # skip free-space medium
    key = _match_itu(mat_name)
    eps_r, sigma, S, xpd, thick = _ITU_DB.get(key, _DEFAULT_MAT)
    matched = key if key else 'DEFAULT'

    # Set EM properties (Sionna may override eps_r/sigma from ITU tables)
    try: mat.relative_permittivity = eps_r
    except: pass
    try: mat.conductivity = sigma
    except: pass

    # Set scattering coefficient
    _S_set = False
    for _attr in ('scattering_coefficient', 'scattering_coeff'):
        if hasattr(mat, _attr):
            try:
                setattr(mat, _attr, float(S))
                if ENABLE_GRAD:
                    dr.enable_grad(getattr(mat, _attr))
                _S_set = True
            except: pass
            break

    # Set XPD coefficient
    _xpd_set = False
    if hasattr(mat, 'xpd_coefficient'):
        try:
            mat.xpd_coefficient = float(xpd)
            if ENABLE_GRAD:
                dr.enable_grad(mat.xpd_coefficient)
            _xpd_set = True
        except: pass

    # Set thickness (penetration loss)
    _thick_set = False
    if hasattr(mat, 'thickness'):
        try:
            mat.thickness = float(thick)
            if ENABLE_GRAD:
                dr.enable_grad(mat.thickness)
            _thick_set = True
        except: pass

    flags = []
    if not _S_set:     flags.append('S:N/A')
    if not _xpd_set:   flags.append('xpd:N/A')
    if not _thick_set: flags.append('thick:N/A')
    if ENABLE_GRAD:    flags.append('grad')
    note = ' '.join(flags) if flags else 'OK'

    print(f'  {mat_name:<28} {eps_r:>6.2f} {sigma:>8.4f} {S:>5.2f} {xpd:>5.2f} {thick:>7.3f}  {note} (match={matched})')

print('=' * 78)
print()
print('  Effect of changing parameters:')
print('  ┌──────────┬────────────────────────────────────────────────────┐')
print('  │ S ↑      │ More diffuse scatter → smoother coverage, less peak│')
print('  │ S ↓      │ More specular → stronger reflections, sharper null │')
print('  │ thick ↑  │ More penetration loss → stronger indoor wall effect │')
print('  │ eps_r ↑  │ Stronger reflection amplitude                      │')
print('  │ sigma ↑  │ More absorption → shorter range per bounce          │')
print('  └──────────┴────────────────────────────────────────────────────┘')
print()
print('  After editing: re-run this cell, then re-run CELL 8 to see new RSSI map.')
print('Done.')


## CELL 6 · Load Transmitter (GPS → local XY, ray-cast Z)

In [ ]:
import time as _time

# ── [1/4] Clear previous transmitters ────────────────────────────────────────
print('[1/4] Clearing previous transmitters ...')
for nm in list(scene.transmitters.keys()):
    scene.remove(nm)
print('  ✓ Cleared')

# ── [2/4] Load TX CSV or use scene centre ─────────────────────────────────────
print('\n[2/4] Loading transmitter ...')
transmitters = []
_t0 = _time.time()

if os.path.exists(TX_CSV):
    df_tx = pd.read_csv(TX_CSV)
    print(f'  ✓ Loaded {len(df_tx)} TX from {TX_CSV}')
    for i, row in df_tx.iterrows():
        lon    = float(row['lon']); lat = float(row['lat'])
        tx_agl = float(row.get('height', TX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        ground_z = ray_cast_ground_z(x, y)
        z  = ground_z + tx_agl
        nm = str(row.get('name', f'tx{i:04d}'))
        tx = Transmitter(name=nm, position=(float(x), float(y), float(z)))
        scene.add(tx); transmitters.append(tx)
else:
    # No CSV — use Ofcom-calibrated TX_LON/TX_LAT from CELL 0c
    print(f'  ⚠ TX CSV not found – using Ofcom site coords TX_LON={TX_LON}, TX_LAT={TX_LAT}')
    x, y, _ = gps_to_local(TX_LON, TX_LAT)
    ground_z = ray_cast_ground_z(x, y)
    z  = ground_z + TX_AGL_M
    tx = Transmitter(name='tx_ofcom', position=(float(x), float(y), float(z)))
    scene.add(tx); transmitters.append(tx)

# Reference TX for downstream cells
tx     = transmitters[0]
abs_z  = _safe(tx.position[2])
tx_agl = TX_AGL_M

# ── [3/4] Antenna arrays ──────────────────────────────────────────────────────
print('\n[3/4] Configuring antenna arrays ...')
scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='iso', polarization='V')
print('  ✓ TX: 1x1 isotropic V-pol')
print('  ✓ RX: 1x1 isotropic V-pol')

# ── [4/4] Summary ─────────────────────────────────────────────────────────────
print('\n[4/4] Transmitter summary:')
for t in transmitters:
    x  = _safe(t.position[0])
    y  = _safe(t.position[1])
    z  = _safe(t.position[2])
    lon, lat = local_to_gps(x, y)
    print(f'  ✓ TX "{t.name}"  GPS=({lon:.5f},{lat:.5f})  '
          f'XY=({x:.1f},{y:.1f})  Z={z:.2f} m  '
          f'AGL={TX_AGL_M:.1f} m  EIRP={EIRP_DBM:.1f} dBm')
print(f'\n  Done in {_time.time()-_t0:.2f}s')


## CELL 7 · Load Receivers (GPS → local XY, ray-cast Z)

In [ ]:
import time as _time

# ── [1/5] Load CSV ───────────────────────────────────────────────────────────
print('[1/5] Loading receiver CSV ...')
if not os.path.exists(RX_CSV):
    print(f'  ✗ RX CSV not found at {RX_CSV}')
    df_rx = None
else:
    df_rx = pd.read_csv(RX_CSV)
    print(f'  ✓ Loaded {len(df_rx)} receivers')

# ── [2/5] Clear previous receivers ───────────────────────────────────────────
print('\n[2/5] Clearing previous receivers ...')
for nm in list(scene.receivers.keys()):
    scene.remove(nm)
print(f'  ✓ Cleared')

# ── [3/5] Convert coordinates ─────────────────────────────────────────────────
print('\n[3/5] Converting coordinates and computing ground heights ...')
receivers = []
_t0 = _time.time()

if df_rx is not None:
    for i, row in df_rx.iterrows():
        lon = float(row['lon']); lat = float(row['lat'])
        agl = float(row.get('height', RX_AGL_M))
        x, y, _ = gps_to_local(lon, lat)
        ground_z = ray_cast_ground_z(x, y)
        z = ground_z + agl
        nm = str(row.get('name', f'RX_{i+1:04d}'))
        rx = Receiver(name=nm, position=(float(x), float(y), float(z)))
        rx._ground_z = ground_z
        rx._agl      = agl
        scene.add(rx)
        receivers.append(rx)
    print(f'  ✓ Loaded {len(receivers)} receivers in {_time.time()-_t0:.2f}s')
else:
    rx = Receiver(name='rx0', position=(100.0, 0.0, RX_AGL_M))
    rx._ground_z = 0.0; rx._agl = RX_AGL_M
    scene.add(rx); receivers.append(rx)
    print('  ✓ Default single receiver placed')

# ── [4/5] Validation ─────────────────────────────────────────────────────────
print('\n[4/5] Validation (first 5 receivers):')
for rx in receivers[:5]:
    x  = _safe(rx.position[0])
    y  = _safe(rx.position[1])
    z  = _safe(rx.position[2])
    gz = getattr(rx, '_ground_z', z - getattr(rx, '_agl', RX_AGL_M))
    lon, lat = local_to_gps(x, y)
    print(f'  {rx.name}: XY({x:.1f}, {y:.1f})  Z={z:.2f}  '
          f'(ground={gz:.2f})  GPS=({lon:.5f},{lat:.5f})')

# ── [5/5] Bbox containment ────────────────────────────────────────────────────
print('\n[5/5] Bbox containment:')
try:
    _bbox  = scene.mi_scene.bbox()
    xs = [_safe(r.position[0]) for r in receivers]
    ys = [_safe(r.position[1]) for r in receivers]
    zs = [_safe(r.position[2]) for r in receivers]
    x_ok = all(float(_bbox.min[0]) <= x <= float(_bbox.max[0]) for x in xs)
    y_ok = all(float(_bbox.min[1]) <= y <= float(_bbox.max[1]) for y in ys)
    z_ok = all(float(_bbox.min[2]) <= z <= float(_bbox.max[2]) for z in zs)
    print(f'  All X inside: {x_ok},  Y inside: {y_ok},  Z inside: {z_ok}')
    if not (x_ok and y_ok):
        print('  ⚠ Some receivers outside scene bbox — check GPS coordinates')
except Exception as _e:
    print(f'  Bbox check skipped: {_e}')

print(f'\nScene: {len(scene.transmitters)} TX,  {len(scene.receivers)} RX')


## CELL 8 · Coverage Map (Sionna 0.19)

In [ ]:
# ====================================================================
# CELL 8 — COVERAGE MAP (WITH & WITHOUT SCATTER)  [Sionna 0.19]
# ====================================================================
import numpy as np, os, time

print('=' * 70)
print('CELL 8 — COVERAGE MAP SOLVER')
print('=' * 70)

bbox   = scene.mi_scene.bbox()
margin = 50.0
gx_min = float(bbox.min[0]) - margin
gx_max = float(bbox.max[0]) + margin
gy_min = float(bbox.min[1]) - margin
gy_max = float(bbox.max[1]) + margin
center_x = (gx_min + gx_max) / 2
center_y = (gy_min + gy_max) / 2
ground_z_at_center = ray_cast_ground_z(center_x, center_y)
center_z = ground_z_at_center + RX_AGL_M
nx = int((gx_max - gx_min) / GRID_SIZE_M)
ny = int((gy_max - gy_min) / GRID_SIZE_M)
print(f'Grid: {nx} x {ny} cells  ({GRID_SIZE_M} m res)  map Z={center_z:.2f} m')

# ── Scene sanity check ───────────────────────────────────────────────────────
print(f'\n[DIAG] TX count      : {len(scene.transmitters)}')
for _n, _t in scene.transmitters.items():
    print(f'       TX "{_n}"  pos={[round(_safe(_t.position[i]),2) for i in range(3)]}')
print(f'[DIAG] CM center_z   : {center_z:.2f} m')
print(f'[DIAG] Grid          : {nx} x {ny}  gx=[{gx_min:.0f},{gx_max:.0f}]  gy=[{gy_min:.0f},{gy_max:.0f}]')

# ── Helper: path_gain -> (rssi_dBm, path_loss_dB) ────────────────────────────
def _cm_to_dbm(cm):
    """Return (rssi_db, path_loss_db) arrays from a Sionna 0.19 CoverageMap."""
    arr = np.array(cm.path_gain)
    if arr.ndim == 3:   arr = arr[0]       # (num_tx, ny, nx) -> (ny, nx)
    elif arr.ndim == 4: arr = arr[0, 0]    # edge-case extra dim
    path_loss_db = -10.0 * np.log10(np.maximum(arr, 1e-20))
    # Use EIRP_DBM directly — tx.power_dbm is Sionna internal default, not our EIRP
    rssi_db = EIRP_DBM - path_loss_db + SYS_GAIN  # EIRP - PL + system gain (Ofcom: 54 - PL + 16)
    return rssi_db, path_loss_db

# ── GPU/CPU auto-fallback wrapper ─────────────────────────────────────────────
import gc as _gc

def _compute_cm(scattering, label):
    """Run coverage_map on GPU; auto-retry on CPU if OOM."""
    _cfg = dict(
        cm_center      = (center_x, center_y, center_z),
        cm_orientation = (0., 0., 0.),
        cm_size        = (gx_max - gx_min, gy_max - gy_min),
        cm_cell_size   = (GRID_SIZE_M, GRID_SIZE_M),
        max_depth      = MAX_DEPTH,
        num_samples    = NUM_SAMPLES_CM,
        los=True, reflection=True, scattering=scattering, diffraction=True)
    print(f'\nComputing coverage map {label} ...')
    t0 = time.time()

    # Variant is fixed at load time in CELL 4 (FORCE_CPU_RT sets llvm_ad_rgb
    # before load_scene). Switching variant after scene load breaks Mitsuba
    # internal types — do NOT call mi.set_variant() here.
    print(f'  Mitsuba variant: {mi.variant()}')

    try:
        cm = scene.coverage_map(**_cfg)
        print(f'  Done in {time.time()-t0:.1f}s')
        return cm
    except Exception as _oom:
        _msg = str(_oom).lower()
        if any(k in _msg for k in ['oom','resource exhausted','memory','incompatible']):
            print(f'  ⚠ Error: {str(_oom)[:120]}')
            print('  → Set FORCE_CPU_RT=True in CELL 0c and restart kernel + re-run from CELL 4')
        raise

# ── WITH scattering ───────────────────────────────────────────────────────────
cm_scatter    = _compute_cm(True,  'WITH scattering')

# ── WITHOUT scattering ────────────────────────────────────────────────────────
cm_no_scatter = _compute_cm(False, 'WITHOUT scattering')

# ── Decode to numpy ──────────────────────────────────────────────────────────
rssi_scatter,    path_loss_scatter    = _cm_to_dbm(cm_scatter)
_raw_pg = np.array(cm_scatter.path_gain)
print(f'[DIAG] path_gain shape: {_raw_pg.shape}  dtype={_raw_pg.dtype}')
print(f'[DIAG] path_gain raw: min={_raw_pg.min():.2e}  max={_raw_pg.max():.2e}  mean={_raw_pg.mean():.2e}')
print(f'[DIAG] non-zero cells: {(_raw_pg > 1e-30).sum()} / {_raw_pg.size}')
rssi_no_scatter, path_loss_no_scatter = _cm_to_dbm(cm_no_scatter)

# NOISE_FLOOR_DBM = NOISE_FLOOR from CELL 0c (-109 dBm, Ofcom system noise floor)
for label, r, pl in [
    ('With scatter',    rssi_scatter,    path_loss_scatter),
    ('Without scatter', rssi_no_scatter, path_loss_no_scatter),
]:
    covered = r > NOISE_FLOOR_DBM          # cells with meaningful signal
    n_cov   = int(covered.sum())
    n_total = int(r.size)
    v  = r[covered]
    p  = pl[covered]
    print(f'{label:20s}')
    print(f'  Coverage: {n_cov:,}/{n_total:,} cells ({100*n_cov/n_total:.1f}%)')
    if n_cov:
        print(f'  RSSI (covered): mean={v.mean():.1f}  std={v.std():.1f}  '
              f'min={v.min():.1f}  max={v.max():.1f} dBm')
        print(f'  PL   (covered): mean={p.mean():.1f}  std={p.std():.1f}  '
              f'min={p.min():.1f}  max={p.max():.1f} dB')
    else:
        print(f'  ⚠ No covered cells — increase NUM_SAMPLES_CM in CELL 0c')

# Scatter impact on covered cells
cov_both = (rssi_scatter > NOISE_FLOOR_DBM) & (rssi_no_scatter > NOISE_FLOOR_DBM)
if cov_both.sum() > 0:
    delta = rssi_scatter[cov_both] - rssi_no_scatter[cov_both]
    print(f'\nScatter impact (covered cells): '
          f'mean={delta.mean():.2f}  std={delta.std():.2f}  '
          f'max={delta.max():.2f} dB')
    print(f'  mean~0 is correct physics: scatter redistributes energy (some cells +, some -)')
    print(f'  std={delta.std():.1f} dB = per-cell variability from diffuse scatter')


## CELL 9 · Interpolate to Receiver Positions

In [ ]:
# ====================================================================
# CELL 9 — INTERPOLATE COVERAGE MAP TO RECEIVER POSITIONS
# ====================================================================
from scipy.spatial import KDTree
import pandas as pd, numpy as np, os

print('=' * 70)
print('CELL 9 — INTERPOLATING RADIO MAPS TO RX POSITIONS')
print('=' * 70)

# Grid centres (must match CELL 8 grid)
x_centers = np.linspace(gx_min, gx_max, nx)
y_centers  = np.linspace(gy_min, gy_max, ny)
X, Y = np.meshgrid(x_centers, y_centers)
grid_points = np.column_stack([X.ravel(), Y.ravel()])
tree = KDTree(grid_points)

# Receiver XY positions (local UTM, metres)
rx_coords = np.array([(_safe(rx.position[0]), _safe(rx.position[1])) for rx in receivers])
distances, indices = tree.query(rx_coords)

def interpolate_and_save(rssi_map, path_loss_map, suffix):
    rssi_at_rx = rssi_map.ravel()[indices]
    pl_at_rx   = path_loss_map.ravel()[indices]
    # Convert to GPS for RMSE comparison with Ofcom drive test CSV
    lon_lat    = [local_to_gps(x, y) for x, y in rx_coords]
    df = pd.DataFrame({
        'receiver':     [rx.name for rx in receivers],
        'lon':          [ll[0] for ll in lon_lat],
        'lat':          [ll[1] for ll in lon_lat],
        'x_m':          rx_coords[:, 0],
        'y_m':          rx_coords[:, 1],
        'z_m':          [_safe(rx.position[2]) for rx in receivers],
        'rssi_dbm':     rssi_at_rx,
        'path_loss_db': pl_at_rx,
        'grid_dist_m':  distances,
    })
    out = os.path.join(OUT_DIR, f'receiver_results_{suffix}.csv')
    df.to_csv(out, index=False)
    print(f'  Saved {len(df)} receivers -> {out}')
    return df

df_scatter    = interpolate_and_save(rssi_scatter,    path_loss_scatter,    'with_scatter')
df_no_scatter = interpolate_and_save(rssi_no_scatter, path_loss_no_scatter, 'without_scatter')

for label, df in [('With scatter', df_scatter), ('Without scatter', df_no_scatter)]:
    print(f'{label:20s}: RSSI mean={df.rssi_dbm.mean():.1f}  std={df.rssi_dbm.std():.1f} dBm  '
          f'| PL  mean={df.path_loss_db.mean():.1f}  std={df.path_loss_db.std():.1f} dB')


## CELL 10 · Coverage Map Visualisation

In [ ]:
# ====================================================================
# CELL 10 — COVERAGE MAPS (WITH vs WITHOUT SCATTER)
# ====================================================================
import matplotlib.pyplot as plt, os

print('=' * 70)
print('CELL 10 — PLOTTING COVERAGE MAPS')
print('=' * 70)

tx   = list(scene.transmitters.values())[0]
tx_x = _safe(tx.position[0]); tx_y = _safe(tx.position[1])

# Sample receivers for overlay (every 200th for readability)
sample_step = max(1, len(receivers) // 200)
rx_plot_x = [_safe(r.position[0]) for r in receivers[::sample_step]]
rx_plot_y = [_safe(r.position[1]) for r in receivers[::sample_step]]

vmin, vmax = -120, -40
ext = [gx_min, gx_max, gy_min, gy_max]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, data, title in [
    (axes[0], rssi_scatter,    f'With scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
    (axes[1], rssi_no_scatter, f'Without scattering @ {FREQUENCY_HZ/1e9:.2f} GHz'),
]:
    im = ax.imshow(data, origin='lower', extent=ext,
                   cmap='jet', aspect='auto', vmin=vmin, vmax=vmax)
    ax.scatter(tx_x, tx_y, marker='*', s=300, c='gold', edgecolors='black', label='TX')
    ax.scatter(rx_plot_x, rx_plot_y, s=5, c='cyan', alpha=0.5, label='RX')
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_title(title); ax.legend(loc='upper right', fontsize=8)
    plt.colorbar(im, ax=ax, label='RSSI (dBm)')

plt.suptitle(f'Coverage Map | EIRP={EIRP_DBM:.1f} dBm | {FREQUENCY_HZ/1e9:.2f} GHz', fontsize=13)
plt.tight_layout()
out = os.path.join(OUT_DIR, 'coverage_map_comparison.png')
plt.savefig(out, dpi=150); plt.show()
print(f'Saved -> {out}')


## CELL 10a · RSSI / Path Loss / SINR Comparison

In [ ]:
# ====================================================================
# CELL 10a — RSSI / PATH LOSS / SINR COMPARISON (WITH vs WITHOUT SCATTER)
# ====================================================================
import matplotlib.pyplot as plt, numpy as np, os

print('=' * 70)
print('CELL 10a — PROPAGATION COMPARISON')
print('=' * 70)

# ── Noise floor and SINR ─────────────────────────────────────────────────────
NF_DB        = 5.0       # noise figure (dB)
kT_dBm_Hz    = -174.0   # thermal noise PSD (dBm/Hz)
noise_dbm    = kT_dBm_Hz + 10 * np.log10(BANDWIDTH_HZ) + NF_DB
print(f'Noise floor: {noise_dbm:.1f} dBm  (BW={BANDWIDTH_HZ/1e6:.0f} MHz, NF={NF_DB} dB)')

sinr_scatter    = rssi_scatter    - noise_dbm
sinr_no_scatter = rssi_no_scatter - noise_dbm

ext = [gx_min, gx_max, gy_min, gy_max]
tx  = list(scene.transmitters.values())[0]
tx_x = _safe(tx.position[0]); tx_y = _safe(tx.position[1])
sample_step = max(1, len(receivers) // 200)
rx_plot_x = [_safe(r.position[0]) for r in receivers[::sample_step]]
rx_plot_y = [_safe(r.position[1]) for r in receivers[::sample_step]]

# ── 3x2 map grid ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
for ax, data, title, cmap, vmin, vmax, unit in [
    (axes[0, 0], rssi_scatter,        'RSSI – With scatter',        'jet',      -120, -40, 'dBm'),
    (axes[0, 1], rssi_no_scatter,     'RSSI – No scatter',          'jet',      -120, -40, 'dBm'),
    (axes[1, 0], path_loss_scatter,   'Path Loss – With scatter',   'plasma_r',   60, 140, 'dB'),
    (axes[1, 1], path_loss_no_scatter,'Path Loss – No scatter',     'plasma_r',   60, 140, 'dB'),
    (axes[2, 0], sinr_scatter,        'SINR – With scatter',        'RdYlGn',    -10,  30, 'dB'),
    (axes[2, 1], sinr_no_scatter,     'SINR – No scatter',          'RdYlGn',    -10,  30, 'dB'),
]:
    im = ax.imshow(data, origin='lower', extent=ext,
                   cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.scatter(tx_x, tx_y, marker='*', s=200, c='gold', edgecolors='black')
    ax.scatter(rx_plot_x, rx_plot_y, s=3, c='cyan', alpha=0.5, label='RX')
    ax.set_title(title); ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.legend(loc='upper right', fontsize=7)
    plt.colorbar(im, ax=ax, label=unit)

plt.suptitle('Propagation Comparison', fontsize=14)
plt.tight_layout()
out = os.path.join(OUT_DIR, 'propagation_comparison.png')
plt.savefig(out, dpi=150); plt.show()
print(f'Saved -> {out}')

# ── Difference map: RSSI(scatter) – RSSI(no scatter) ─────────────────────────
rssi_diff = rssi_scatter - rssi_no_scatter
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(rssi_diff, origin='lower', extent=ext,
               cmap='RdBu_r', vmin=-5, vmax=5)
ax.scatter(tx_x, tx_y, marker='*', s=200, c='gold', edgecolors='black', label='TX')
ax.scatter(rx_plot_x, rx_plot_y, s=5, c='cyan', alpha=0.5, label='RX')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Impact of scattering: ΔRSSI = With – Without (dB)')
plt.colorbar(im, label='dB'); ax.legend()
plt.tight_layout()
out2 = os.path.join(OUT_DIR, 'scattering_impact_map.png')
plt.savefig(out2, dpi=150); plt.show()
print(f'Saved -> {out2}')

# ── Per-receiver scatter comparison ──────────────────────────────────────────
rx_rssi_scatter    = df_scatter['rssi_dbm'].values
rx_rssi_no_scatter = df_no_scatter['rssi_dbm'].values
rx_pl_scatter      = df_scatter['path_loss_db'].values
rx_pl_no_scatter   = df_no_scatter['path_loss_db'].values

diff_rx = rx_rssi_scatter - rx_rssi_no_scatter
fig, axes2 = plt.subplots(1, 2, figsize=(12, 5))
axes2[0].scatter(rx_rssi_no_scatter, rx_rssi_scatter, alpha=0.5, s=15)
axes2[0].plot([-140, -20], [-140, -20], 'k--', label='1:1 line')
axes2[0].set_xlabel('RSSI no scatter (dBm)'); axes2[0].set_ylabel('RSSI with scatter (dBm)')
axes2[0].set_title('RSSI at receiver positions'); axes2[0].legend(); axes2[0].grid(True)
axes2[1].hist(diff_rx, bins=30, edgecolor='black', alpha=0.7)
axes2[1].axvline(0, color='r', linestyle='--', label='Zero diff')
axes2[1].set_xlabel('ΔRSSI = with − without (dB)'); axes2[1].set_ylabel('Count')
axes2[1].set_title(f'Scatter impact: mean={np.mean(diff_rx):.2f} dB  std={np.std(diff_rx):.2f} dB')
axes2[1].legend()
plt.tight_layout()
out3 = os.path.join(OUT_DIR, 'receiver_scatter_comparison.png')
plt.savefig(out3, dpi=150); plt.show()
print(f'Saved -> {out3}')

for label, r in [('With scatter', rx_rssi_scatter), ('Without scatter', rx_rssi_no_scatter)]:
    print(f'{label:20s}: mean={r.mean():.1f}  std={r.std():.1f}  '
          f'min={r.min():.1f}  max={r.max():.1f} dBm')


## CELL 9b · Path Computation (Sionna 0.19)

In [ ]:
# ====================================================================
# CELL 9b — PATH SOLVER WITH PER-RAY EXTRACTION  [Sionna 0.19]
# ====================================================================
# Sionna 0.19 API: scene.compute_paths() replaces PathSolver().
#   - reflection  = specular_reflection (2.0)
#   - scattering  = diffuse_reflection  (2.0)
# Coverage strategy:
#   - ALL receivers processed (no distance filter)
#   - Adaptive samples: scale with distance so far receivers still get hits
#       base 10M rays; ×4 beyond 5 km; ×8 beyond 9 km
#   - Auto-retry: batches with 0 paths retried at 4× samples (max 2 retries)
#   - diffraction=True to model urban NLOS paths
# ====================================================================
import gc, time, os, numpy as np, pandas as pd, drjit as dr
from datetime import datetime

print('=' * 70)
print('CELL 9b — PATH SOLVER  [Sionna 0.19]')
print('=' * 70)

# ── Configuration ─────────────────────────────────────────────────────────────
SAVE_PER_RAY    = True
MAX_RAYS_PER_RX = 300
BATCH_SIZE      = 50
C               = 3e8

PS_CONFIG_BASE = dict(
    max_depth   = MAX_DEPTH,
    los         = True,
    reflection  = True,
    scattering  = True,
    diffraction = True,
)

tx             = list(scene.transmitters.values())[0]
tx_pwr_dbm     = EIRP_DBM
tx_pos         = np.array([_safe(tx.position[0]),
                            _safe(tx.position[1]),
                            _safe(tx.position[2])])
rx_gain_total  = SYS_GAIN

print(f'  TX EIRP         : {tx_pwr_dbm:.1f} dBm')
print(f'  TX position     : ({tx_pos[0]:.1f}, {tx_pos[1]:.1f}, {tx_pos[2]:.1f}) m')
print(f'  RX gain total   : {rx_gain_total:.1f} dB')
print(f'  Batch size      : {BATCH_SIZE} receivers')
print(f'  Base num_samples: {NUM_SAMPLES_PS:,}')
print()
for k, v in PS_CONFIG_BASE.items():
    print(f'  {k:15s}: {v}')

# ── Adaptive sample count based on TX-RX distance ────────────────────────────
def adaptive_samples(dist_m):
    """Scale ray count with distance so far receivers get enough hits."""
    if dist_m > 9000:
        return NUM_SAMPLES_PS * 8
    elif dist_m > 5000:
        return NUM_SAMPLES_PS * 4
    else:
        return NUM_SAMPLES_PS

# ── CIR helpers ───────────────────────────────────────────────────────────────
def extract_amplitudes(paths):
    a = paths.a
    if isinstance(a, tuple):
        a_np = a[0].numpy() + 1j * a[1].numpy()
    else:
        a_np = np.array(a)
    a_np = np.squeeze(a_np)
    while a_np.ndim > 2:
        a_np = a_np[..., 0]
    if a_np.ndim == 1:
        a_np = a_np[np.newaxis, :]
    return a_np

def extract_tau(paths, num_rx, num_paths):
    tau = getattr(paths, 'tau', None)
    if tau is None:
        return np.full((num_rx, num_paths), np.nan, np.float32)
    try:
        tau_np = tau.numpy() if hasattr(tau, 'numpy') else np.array(tau)
        tau_np = np.squeeze(tau_np)
        while tau_np.ndim > 2:
            tau_np = tau_np[..., 0]
        if tau_np.ndim == 1:
            tau_np = tau_np[np.newaxis, :]
        return tau_np
    except Exception:
        return np.full((num_rx, num_paths), np.nan, np.float32)

def summary_metrics(a_row):
    pwr   = np.abs(a_row) ** 2
    valid = pwr > 1e-30
    if not np.any(valid):
        return np.nan, np.nan, np.nan, 0
    pv = pwr[valid]; av = a_row[valid]
    best_pl       = -10 * np.log10(np.max(pv))
    incoherent_pl = -10 * np.log10(np.sum(pv))
    coh_pwr       = np.abs(np.sum(av)) ** 2
    coherent_pl   = -10 * np.log10(coh_pwr) if coh_pwr > 1e-30 else np.nan
    return best_pl, incoherent_pl, coherent_pl, int(np.sum(valid))

def ray_type_heuristic(path_len, los_dist, pwr, max_pwr):
    if los_dist > 0 and abs(path_len - los_dist) / los_dist < 0.01:
        return 'LOS'
    ratio  = pwr / max_pwr if max_pwr > 0 else 0
    excess = path_len - los_dist
    if excess < 50  and ratio > 0.01:  return 'REFLECTION'
    if excess >= 50 and ratio > 0.001: return 'MULTI_REFLECTION'
    if ratio < 0.01:                   return 'DIFFRACTION'
    if ratio < 0.001:                  return 'SCATTERING'
    return 'UNKNOWN'

def run_batch(batch, cfg):
    for name in list(scene.receivers.keys()):
        scene.remove(name)
    for rx in batch:
        scene.add(rx)
    try:
        paths = scene.compute_paths(**cfg)
    except Exception as _oom:
        if any(k in str(_oom).lower() for k in ['oom','resource exhausted','memory']):
            _reduced = {**cfg, 'num_samples': 50_000}
            paths = scene.compute_paths(**_reduced)
        else:
            raise
    return paths

# ── ALL receivers — no distance filter ───────────────────────────────────────
_all_rx = list(receivers)
total   = len(_all_rx)

# Sort by distance so nearby batches are processed first (faster early results)
tx_pos2d = tx_pos[:2]
_all_rx.sort(key=lambda rx: float(np.linalg.norm(
    [_safe(rx.position[0]) - tx_pos2d[0], _safe(rx.position[1]) - tx_pos2d[1]])))

ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_csv = os.path.join(OUT_DIR, f'path_solver_summary_{ts}.csv')
per_ray_csv = os.path.join(OUT_DIR, f'path_solver_per_ray_{ts}.csv') if SAVE_PER_RAY else None

summary_rows = []
per_ray_rows = []
errors       = 0
t0           = time.time()

print(f'\nProcessing ALL {total} receivers in batches of {BATCH_SIZE} ...')
print('Adaptive samples: ×1 <5km, ×4 5-9km, ×8 >9km')
print('Auto-retry: 0-path batches retried at 4× samples (max 2 attempts)\n')

for b_start in range(0, total, BATCH_SIZE):
    batch    = _all_rx[b_start : b_start + BATCH_SIZE]
    # Use max distance in batch to set sample count
    max_dist = max(float(np.linalg.norm(
        [_safe(rx.position[0]) - tx_pos2d[0], _safe(rx.position[1]) - tx_pos2d[1]]))
        for rx in batch)
    n_samp   = adaptive_samples(max_dist)
    cfg      = {**PS_CONFIG_BASE, 'num_samples': n_samp}

    paths         = None
    batch_paths   = 0
    retry         = 0
    while retry <= 2:
        try:
            paths = run_batch(batch, cfg)
            a_all = extract_amplitudes(paths)
            batch_paths = int(np.sum(np.abs(a_all) ** 2 > 1e-30))
            if batch_paths == 0 and retry < 2:
                del paths, a_all; gc.collect()
                cfg = {**cfg, 'num_samples': cfg['num_samples'] * 4}
                retry += 1
                print(f'  [RETRY {retry}] batch {b_start}: 0 paths — retrying with {cfg["num_samples"]:,} samples')
                continue
            break
        except Exception as _e:
            print(f'  [WARN] Batch {b_start}: {_e}')
            paths = None; a_all = None; batch_paths = 0
            break

    if paths is None or batch_paths == 0:
        for rx in batch:
            los_dist = float(np.linalg.norm(
                np.array([_safe(rx.position[0]),_safe(rx.position[1]),_safe(rx.position[2])]) - tx_pos))
            summary_rows.append({'receiver': rx.name,
                'x_m': _safe(rx.position[0]), 'y_m': _safe(rx.position[1]),
                'z_m': _safe(rx.position[2]), 'dist_from_tx_m': los_dist,
                'num_samples_used': n_samp, 'num_paths': 0,
                'path_loss_best_db': np.nan, 'path_loss_incoherent_db': np.nan,
                'path_loss_coherent_db': np.nan, 'rssi_best_dbm': np.nan,
                'rssi_incoherent_dbm': np.nan, 'rssi_coherent_dbm': np.nan})
        if paths is None:
            errors += len(batch)
        del paths; gc.collect()
        done = min(b_start + BATCH_SIZE, total)
        if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
            elapsed = time.time() - t0
            print(f'  [{done}/{total}]  {elapsed:.0f}s  (0 paths — NLOS or too far)')
        continue

    n_b, n_p = a_all.shape
    tau_all  = extract_tau(paths, n_b, n_p)

    for i, rx in enumerate(batch):
        rx_pos   = np.array([_safe(rx.position[0]),
                              _safe(rx.position[1]),
                              _safe(rx.position[2])])
        los_dist = float(np.linalg.norm(rx_pos - tx_pos))
        idx      = i if i < n_b else n_b - 1

        best_pl, incoh_pl, coh_pl, n_valid = summary_metrics(a_all[idx])

        summary_rows.append({
            'receiver'                : rx.name,
            'x_m'                    : _safe(rx.position[0]),
            'y_m'                    : _safe(rx.position[1]),
            'z_m'                    : _safe(rx.position[2]),
            'dist_from_tx_m'         : los_dist,
            'num_samples_used'       : n_samp,
            'num_paths'              : n_valid,
            'path_loss_best_db'      : best_pl,
            'path_loss_incoherent_db': incoh_pl,
            'path_loss_coherent_db'  : coh_pl,
            'rssi_best_dbm'          : (tx_pwr_dbm - best_pl  + rx_gain_total) if not np.isnan(best_pl)  else np.nan,
            'rssi_incoherent_dbm'    : (tx_pwr_dbm - incoh_pl + rx_gain_total) if not np.isnan(incoh_pl) else np.nan,
            'rssi_coherent_dbm'      : (tx_pwr_dbm - coh_pl   + rx_gain_total) if not np.isnan(coh_pl)   else np.nan,
        })

        if SAVE_PER_RAY and n_valid > 0:
            a_row   = a_all[idx]
            pwr_row = np.abs(a_row) ** 2
            order   = np.argsort(pwr_row)[::-1]
            max_pwr = pwr_row[order[0]]
            strong_phase = np.angle(a_row[order[0]], deg=True)
            for rank, ray_i in enumerate(order[:MAX_RAYS_PER_RX]):
                ac      = a_row[ray_i]
                pwr_ray = float(pwr_row[ray_i])
                if pwr_ray <= 1e-30:
                    break
                phase   = float(np.angle(ac, deg=True))
                ph_diff = (phase - strong_phase + 180) % 360 - 180
                delay   = float(tau_all[idx, ray_i]) if not np.isnan(tau_all[idx, ray_i]) else np.nan
                plen    = delay * C if not np.isnan(delay) else np.nan
                rtype   = ray_type_heuristic(plen, los_dist, pwr_ray, max_pwr) \
                          if not np.isnan(plen) else 'UNKNOWN'
                per_ray_rows.append({
                    'receiver'       : rx.name,
                    'rank'           : rank,
                    'ray_type'       : rtype,
                    'power_linear'   : pwr_ray,
                    'path_loss_db'   : -10 * np.log10(pwr_ray),
                    'amplitude_real' : float(ac.real),
                    'amplitude_imag' : float(ac.imag),
                    'phase_deg'      : phase,
                    'phase_diff_deg' : ph_diff,
                    'constructive'   : 'STRONGEST' if rank == 0 else
                                       ('CONSTRUCTIVE' if abs(ph_diff) < 90 else 'DESTRUCTIVE'),
                    'delay_s'        : delay,
                    'path_length_m'  : plen,
                })

    del paths, a_all, tau_all
    gc.collect()
    done = min(b_start + BATCH_SIZE, total)
    if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
        elapsed = time.time() - t0
        eta     = (total - done) / max(done / max(elapsed, 0.001), 1e-9)
        print(f'  [{done}/{total}]  {elapsed:.0f}s elapsed  ETA {eta/60:.1f} min', flush=True)

# Restore all receivers
for name in list(scene.receivers.keys()):
    scene.remove(name)
for rx in _all_rx:
    scene.add(rx)

# ── Save ──────────────────────────────────────────────────────────────────────
df_ps = pd.DataFrame(summary_rows)
df_ps.to_csv(summary_csv, index=False)
print(f'\n  Summary  -> {summary_csv}')

if SAVE_PER_RAY and per_ray_rows:
    df_ray = pd.DataFrame(per_ray_rows)
    df_ray.to_csv(per_ray_csv, index=False)
    print(f'  Per-ray  -> {per_ray_csv}  ({len(df_ray):,} rays)')

elapsed = time.time() - t0
valid   = df_ps[df_ps['num_paths'] > 0]
nan_rx  = df_ps[df_ps['num_paths'] == 0]
print(f'\n  Total time      : {elapsed:.1f}s  |  Errors: {errors}')
print(f'  Receivers solved: {len(valid)}/{total} ({100*len(valid)/max(total,1):.1f}%)')
print(f'  Zero-path (NaN) : {len(nan_rx)} — covered by CELL 9c CM fallback')

for col, lbl in [('path_loss_incoherent_db','PL incoherent'),
                 ('path_loss_coherent_db',  'PL coherent'),
                 ('rssi_incoherent_dbm',    'RSSI incoher.')]:
    v = df_ps[col].dropna()
    if len(v):
        print(f'  {lbl:20s}: mean={v.mean():.1f}  std={v.std():.1f}  '
              f'min={v.min():.1f}  max={v.max():.1f}')


## CELL 11 · Path Loss vs Distance

In [ ]:
tx_pos2d = np.array([_safe(tx.position[0]), _safe(tx.position[1])])
dist_m   = np.linalg.norm(df_out[['x_m','y_m']].values - tx_pos2d, axis=1)
df_out['dist_m'] = dist_m
df_out['path_loss'] = -df_out['pg_db']

_lam  = C / FREQUENCY_HZ
d_ref = np.linspace(max(dist_m.min(), 10), dist_m.max(), 300)
fspl  = 20*np.log10(4*np.pi*d_ref / _lam)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(dist_m, df_out['path_loss'], s=5, alpha=0.4, c='steelblue', label='Simulated')
ax.plot(d_ref, fspl, 'r--', lw=2, label='Free-space PL')
ax.set_xlabel('Distance TX→RX (m)'); ax.set_ylabel('Path Loss (dB)')
ax.set_title(f'{CITY_NAME} – Path Loss @ {FREQUENCY_HZ/1e9:.2f} GHz')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'path_loss_vs_distance.png'), dpi=150)
plt.show()
print('All results saved to:', OUT_DIR)

## CELL 11b · TX→RX Distance Analysis

In [ ]:
# ====================================================================
# CELL 11b — TX→RX DISTANCE ANALYSIS
# ====================================================================
# Shows exact distance from TX to every receiver.
# Determines which receivers the path solver can handle (< MAX_DIST_PS_M)
# vs which need coverage-map interpolation (CELL 9).
# ====================================================================
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print('=' * 70)
print('CELL 11b — TX→RX DISTANCE ANALYSIS')
print('=' * 70)

# ── TX position ───────────────────────────────────────────────────────────────
tx      = list(scene.transmitters.values())[0]
tx_x    = _safe(tx.position[0])
tx_y    = _safe(tx.position[1])
tx_z    = _safe(tx.position[2])
tx_lon, tx_lat = local_to_gps(tx_x, tx_y)
print(f'TX position : ({tx_x:.1f}, {tx_y:.1f}, {tx_z:.2f}) m  GPS=({tx_lon:.5f}, {tx_lat:.5f})')
print(f'TX AGL      : {TX_AGL_M:.1f} m')
print()

# ── Compute 2-D and 3-D distances for all receivers ──────────────────────────
rx_data = []
for rx in receivers:
    rx_x = _safe(rx.position[0])
    rx_y = _safe(rx.position[1])
    rx_z = _safe(rx.position[2])
    d2d  = float(np.sqrt((rx_x - tx_x)**2 + (rx_y - tx_y)**2))
    d3d  = float(np.sqrt((rx_x - tx_x)**2 + (rx_y - tx_y)**2 + (rx_z - tx_z)**2))
    rx_lon, rx_lat = local_to_gps(rx_x, rx_y)
    rx_data.append({
        'name'    : rx.name,
        'x_m'    : rx_x,
        'y_m'    : rx_y,
        'z_m'    : rx_z,
        'lon'    : rx_lon,
        'lat'    : rx_lat,
        'dist_2d_m' : d2d,
        'dist_3d_m' : d3d,
    })

df_dist = pd.DataFrame(rx_data)
total   = len(df_dist)

print(f'Total receivers : {total}')
print(f'Distance 2D (m) : min={df_dist.dist_2d_m.min():.0f}  '
      f'mean={df_dist.dist_2d_m.mean():.0f}  '
      f'max={df_dist.dist_2d_m.max():.0f}')
print()

# ── Distance band breakdown ───────────────────────────────────────────────────
bands = [0, 500, 1000, 2000, 3000, 5000, 8000, 12000, 20000]
labels = [f'{bands[i]/1000:.1f}–{bands[i+1]/1000:.1f} km' for i in range(len(bands)-1)]

print(f'  {"Band":<14} {"N RX":>6} {"% total":>8} {"Path solver":>14} {"CM only":>8}')
print('-' * 55)
for i, (lo, hi) in enumerate(zip(bands[:-1], bands[1:])):
    mask = (df_dist.dist_2d_m >= lo) & (df_dist.dist_2d_m < hi)
    n    = int(mask.sum())
    pct  = 100 * n / total
    ps   = '✅ yes' if hi <= MAX_DIST_PS_M else '❌ no'
    cm   = 'needs CM' if hi > MAX_DIST_PS_M else 'optional'
    print(f'  {labels[i]:<14} {n:>6} {pct:>7.1f}%  {ps:>14} {cm:>10}')

n_ps = int((df_dist.dist_2d_m <= MAX_DIST_PS_M).sum())
n_cm = total - n_ps
print()
print(f'  Path solver range (≤{MAX_DIST_PS_M/1000:.1f} km) : {n_ps:>5} receivers  ({100*n_ps/total:.1f}%)')
print(f'  Coverage map only (>{MAX_DIST_PS_M/1000:.1f} km) : {n_cm:>5} receivers  ({100*n_cm/total:.1f}%)')

# ── Closest and furthest receivers ───────────────────────────────────────────
print()
print('Closest 5 receivers:')
for _, r in df_dist.nsmallest(5, 'dist_2d_m').iterrows():
    print(f'  {r["name"]:<12} {r["dist_2d_m"]:>8.0f} m   GPS=({r["lon"]:.5f}, {r["lat"]:.5f})')
print('Furthest 5 receivers:')
for _, r in df_dist.nlargest(5, 'dist_2d_m').iterrows():
    print(f'  {r["name"]:<12} {r["dist_2d_m"]:>8.0f} m   GPS=({r["lon"]:.5f}, {r["lat"]:.5f})')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(df_dist.dist_2d_m / 1000, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(MAX_DIST_PS_M / 1000, color='red', lw=2, linestyle='--',
           label=f'Path solver limit ({MAX_DIST_PS_M/1000:.1f} km)')
ax.set_xlabel('Distance from TX (km)')
ax.set_ylabel('Number of receivers')
ax.set_title('TX→RX Distance Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Cumulative
ax = axes[1]
d_sorted = np.sort(df_dist.dist_2d_m.values) / 1000
cdf = np.arange(1, len(d_sorted)+1) / len(d_sorted) * 100
ax.plot(d_sorted, cdf, color='steelblue', lw=2)
ax.axvline(MAX_DIST_PS_M / 1000, color='red', lw=2, linestyle='--',
           label=f'Path solver limit: {100*n_ps/total:.1f}% of RX')
for km in [1, 2, 3, 5]:
    pct_km = float(100 * (df_dist.dist_2d_m <= km*1000).sum() / total)
    ax.annotate(f'{pct_km:.1f}%', xy=(km, pct_km),
                xytext=(km+0.2, pct_km-5), fontsize=8, color='grey')
ax.set_xlabel('Distance from TX (km)')
ax.set_ylabel('Cumulative % of receivers')
ax.set_title('Cumulative Distance Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_png = os.path.join(OUT_DIR, 'tx_rx_distance_distribution.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')

# ── Save to CSV ───────────────────────────────────────────────────────────────
dist_csv = os.path.join(OUT_DIR, 'rx_distances_from_tx.csv')
df_dist.to_csv(dist_csv, index=False)
print(f'Saved: {dist_csv}')


## CELL 9c · Unified Coverage: Path Solver + CM Fallback

In [ ]:
# ====================================================================
# CELL 9c — UNIFIED COVERAGE: PATH SOLVER + CM FALLBACK (100% RX)
# ====================================================================
# Strategy:
#   1. Load path solver summary (CELL 9b) — has NaN for most far receivers
#   2. Load CM interpolation (CELL 9)    — has values for all 57% covered cells
#   3. Merge: path solver where paths found, CM fallback elsewhere
#   4. For receivers with NaN in both → mark as uncovered (below noise floor)
# Result: maximum possible coverage for RMSE comparison vs Ofcom.
# ====================================================================
import pandas as pd
import numpy as np
import glob
import os

print('=' * 70)
print('CELL 9c — UNIFIED COVERAGE (PATH SOLVER + CM FALLBACK)')
print('=' * 70)

# ── 1. Load path solver summary ───────────────────────────────────────────────
ps_files = sorted(glob.glob(os.path.join(OUT_DIR, 'path_solver_summary_*.csv')))
if not ps_files:
    print('⚠ No path solver summary found. Run CELL 9b first.')
    print('  Using CM-only output for all receivers.')
    df_ps = pd.DataFrame(columns=['receiver','path_loss_best_db',
                                   'path_loss_incoherent_db','path_loss_coherent_db',
                                   'rssi_best_dbm','rssi_incoherent_dbm','rssi_coherent_dbm',
                                   'num_paths'])
else:
    df_ps = pd.read_csv(max(ps_files, key=os.path.getctime))
    print(f'Path solver  : {len(df_ps)} receivers loaded  '
          f'({df_ps["path_loss_incoherent_db"].notna().sum()} with paths found, '
          f'{df_ps["path_loss_incoherent_db"].isna().sum()} NaN)')

# ── 2. Load CM interpolation (with scatter, primary) ─────────────────────────
cm_files = sorted(glob.glob(os.path.join(OUT_DIR, 'receiver_results_with_scatter.csv')))
if not cm_files:
    raise FileNotFoundError('No CM interpolation found. Run CELL 9 first.')
df_cm = pd.read_csv(cm_files[-1])
df_cm = df_cm.rename(columns={
    'rssi_dbm'    : 'cm_rssi_dbm',
    'path_loss_db': 'cm_path_loss_db',
})
print(f'CM coverage  : {len(df_cm)} receivers  '
      f'({(df_cm["cm_rssi_dbm"] > NOISE_FLOOR).sum()} above noise floor, '
      f'{(df_cm["cm_rssi_dbm"] <= NOISE_FLOOR).sum()} below)')

# ── 3. Merge ──────────────────────────────────────────────────────────────────
# Base: CM for all receivers (gives coordinates + CM values)
df = df_cm.copy()

if len(df_ps) > 0:
    df = df.merge(
        df_ps[['receiver','num_paths',
               'path_loss_best_db','path_loss_incoherent_db','path_loss_coherent_db',
               'rssi_best_dbm','rssi_incoherent_dbm','rssi_coherent_dbm']],
        on='receiver', how='left'
    )
else:
    for col in ['num_paths','path_loss_best_db','path_loss_incoherent_db',
                'path_loss_coherent_db','rssi_best_dbm','rssi_incoherent_dbm',
                'rssi_coherent_dbm']:
        df[col] = np.nan

# ── 4. Build unified columns — PS where available, CM fallback elsewhere ──────
def unified(ps_col, cm_col):
    """Use path solver value if valid, else CM fallback."""
    ps  = df[ps_col].values.copy()
    cm  = df[cm_col].values.copy()
    out = np.where(np.isfinite(ps), ps, cm)
    return out

df['unified_path_loss_db'] = unified('path_loss_incoherent_db', 'cm_path_loss_db')
df['unified_rssi_dbm']     = unified('rssi_incoherent_dbm',     'cm_rssi_dbm')

# Source flag
df['source'] = 'uncovered'
df.loc[df['cm_rssi_dbm'] > NOISE_FLOOR, 'source'] = 'cm'
df.loc[df['path_loss_incoherent_db'].notna(), 'source'] = 'path_solver'

# ── 5. Distance from TX ───────────────────────────────────────────────────────
tx_x = _safe(tx.position[0])
tx_y = _safe(tx.position[1])
df['dist_from_tx_m'] = np.sqrt((df['x_m'] - tx_x)**2 + (df['y_m'] - tx_y)**2)

# ── 6. Summary ────────────────────────────────────────────────────────────────
n_ps  = int((df['source'] == 'path_solver').sum())
n_cm  = int((df['source'] == 'cm').sum())
n_unc = int((df['source'] == 'uncovered').sum())
total = len(df)

print()
print(f'  Source breakdown ({total} total receivers):')
print(f'    Path solver  : {n_ps:>5}  ({100*n_ps/total:5.1f}%)  — full per-ray accuracy')
print(f'    CM fallback  : {n_cm:>5}  ({100*n_cm/total:5.1f}%)  — grid interpolation')
print(f'    Uncovered    : {n_unc:>5}  ({100*n_unc/total:5.1f}%)  — below noise floor in both')
print(f'    TOTAL COVERED: {n_ps+n_cm:>5}  ({100*(n_ps+n_cm)/total:5.1f}%)')

covered = df['unified_rssi_dbm'] > NOISE_FLOOR
v = df.loc[covered, 'unified_rssi_dbm']
p = df.loc[covered, 'unified_path_loss_db']
print()
print(f'  Unified RSSI (covered): mean={v.mean():.1f}  std={v.std():.1f}  '
      f'min={v.min():.1f}  max={v.max():.1f} dBm')
print(f'  Unified PL   (covered): mean={p.mean():.1f}  std={p.std():.1f}  '
      f'min={p.min():.1f}  max={p.max():.1f} dB')

# ── 7. Load Ofcom measurements and compute RMSE ───────────────────────────────
print()
print('RMSE vs Ofcom measurements:')
try:
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    # Identify columns
    name_col = next((c for c in ['name','receiver','rx_name'] if c in df_meas.columns), None)
    rssi_col = next((c for c in ['local_measurement_dBm','rssi_dBm','RSSI_dBm'] if c in df_meas.columns), None)
    pl_col   = next((c for c in ['path_loss_dB','path_loss_db'] if c in df_meas.columns), None)

    if name_col and rssi_col:
        df_meas = df_meas[[name_col, rssi_col]].dropna()
        df_meas.columns = ['receiver', 'measured_rssi_dbm']
        if pl_col and pl_col in df_meas.columns:
            pass
        else:
            df_meas['measured_pl_db'] = EIRP_DBM - df_meas['measured_rssi_dbm'] + SYS_GAIN

        df_merged = df.merge(df_meas, on='receiver', how='inner')
        print(f'  Matched {len(df_merged)} receivers with measurements')

        for src_label, mask in [
            ('All covered',   df_merged['unified_rssi_dbm'] > NOISE_FLOOR),
            ('PS only',       df_merged['source'] == 'path_solver'),
            ('CM only',       df_merged['source'] == 'cm'),
        ]:
            sub = df_merged[mask]
            if len(sub) < 2:
                continue
            err_rssi = sub['unified_rssi_dbm'] - sub['measured_rssi_dbm']
            rmse = float(np.sqrt((err_rssi**2).mean()))
            bias = float(err_rssi.mean())
            print(f'  {src_label:<14}: N={len(sub):>5}  RMSE={rmse:5.2f} dB  bias={bias:+.2f} dB')
    else:
        print('  ⚠ Could not identify name/RSSI columns in measurement CSV')
        print(f'  Columns: {list(df_meas.columns)}')
except FileNotFoundError:
    print(f'  ⚠ MEASUREMENT_CSV not found: {MEASUREMENT_CSV}')
except Exception as e:
    print(f'  ⚠ {e}')

# ── 8. Save unified output ────────────────────────────────────────────────────
out_csv = os.path.join(OUT_DIR, 'unified_coverage.csv')
df.to_csv(out_csv, index=False)
print(f'\nSaved: {out_csv}')
print(f'Columns: {list(df.columns)}')


## CELL 12 · Per-Receiver Statistics (Best, Incoherent, Coherent)

In [ ]:
# ====================================================================
# CELL 12 — PER‑RECEIVER STATISTICS (BEST, INCOHERENT, COHERENT)
# ====================================================================
import glob, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("CELL 12 — RMS DELAY SPREAD & K‑FACTOR (all path loss types)")
print("=" * 70)

per_ray_files = glob.glob(os.path.join(OUT_DIR, "path_solver_per_ray_*.csv"))
if not per_ray_files:
    raise FileNotFoundError("No per‑ray CSV found. Run Cell 9b with SAVE_PER_RAY=True")
latest_per_ray = max(per_ray_files, key=os.path.getctime)
df_ray = pd.read_csv(latest_per_ray)
print(f"Loaded {len(df_ray)} rays from {os.path.basename(latest_per_ray)}")

stats = []
for rx_name, group in df_ray.groupby('receiver_name'):
    power = group['power_linear'].values
    phases_deg = group['phase_deg'].values
    phases_rad = np.deg2rad(phases_deg)
    complex_amp = np.sqrt(power) * np.exp(1j * phases_rad)
    total_power = np.sum(power)
    max_power = np.max(power)
    coherent_power = np.abs(np.sum(complex_amp))**2
    if total_power <= 0:
        continue
    pl_incoherent = -10 * np.log10(total_power)
    pl_best       = -10 * np.log10(max_power)
    pl_coherent   = -10 * np.log10(coherent_power) if coherent_power > 0 else np.nan
    delays = group['delay_s'].values
    mean_delay = np.sum(power * delays) / total_power
    rms_delay = np.sqrt(np.sum(power * (delays - mean_delay)**2) / total_power)
    k_lin = max_power / (total_power - max_power + 1e-12)
    k_db = 10 * np.log10(k_lin)
    type_col = None
    if 'path_type' in df_ray.columns:
        type_col = 'path_type'
    elif 'interaction_sequence' in df_ray.columns:
        type_col = 'interaction_sequence'
    if type_col is not None:
        if type_col == 'interaction_sequence':
            types = group[type_col].fillna('').apply(lambda s: s.split('|')[0] if s else 'UNKNOWN')
        else:
            types = group[type_col]
        type_counts = types.value_counts().to_dict()
    else:
        type_counts = {}
    stats.append({
        'receiver': rx_name,
        'num_paths': len(group),
        'path_loss_best_db': pl_best,
        'path_loss_incoherent_db': pl_incoherent,
        'path_loss_coherent_db': pl_coherent,
        'rms_delay_spread_ns': rms_delay * 1e9,
        'k_factor_db': k_db,
        **{f'count_{k}': v for k, v in type_counts.items()}
    })

df_stats = pd.DataFrame(stats).fillna(0)

def extract_number(name):
    match = re.search(r'\d+', name)
    return int(match.group()) if match else 0
df_stats['_sort_key'] = df_stats['receiver'].apply(extract_number)
df_stats = df_stats.sort_values('_sort_key').drop(columns=['_sort_key'])

stats_csv = os.path.join(OUT_DIR, "path_stats_per_rx.csv")
df_stats.to_csv(stats_csv, index=False)
print(f"Saved per‑receiver statistics to {stats_csv}")

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
rms_valid = df_stats['rms_delay_spread_ns'].dropna()
rms_valid = rms_valid[rms_valid > 0]
if len(rms_valid):
    ax[0].plot(np.sort(rms_valid), np.linspace(0, 1, len(rms_valid)), 'b-', lw=2)
    ax[0].set_xlabel('RMS Delay Spread (ns)'); ax[0].set_ylabel('CDF')
    ax[0].set_title('Delay Spread Distribution'); ax[0].grid(True)
k_valid = df_stats['k_factor_db'].replace([np.inf, -np.inf], np.nan).dropna()
if len(k_valid):
    ax[1].plot(np.sort(k_valid), np.linspace(0, 1, len(k_valid)), 'r-', lw=2)
    ax[1].set_xlabel('K‑factor (dB)'); ax[1].set_ylabel('CDF')
    ax[1].set_title('K‑factor Distribution'); ax[1].grid(True)
plt.tight_layout()
cdf_png = os.path.join(OUT_DIR, "rms_k_cdf.png")
plt.savefig(cdf_png, dpi=150); plt.close()
print(f"Saved CDF plot to {cdf_png}")

plt.figure(figsize=(8,5))
plt.hist(df_stats['num_paths'], bins=50, alpha=0.7, color='green')
plt.xlabel('Number of paths per receiver'); plt.ylabel('Frequency')
plt.title('Multipath Count Distribution'); plt.grid(True)
hist_png = os.path.join(OUT_DIR, "num_paths_hist.png")
plt.savefig(hist_png, dpi=150); plt.close()
print(f"Saved histogram to {hist_png}")
print("\n Cell 12 complete")

## CELL 13 · Propagation Mechanism Comparison

In [ ]:
# ====================================================================
# CELL 13 — PROPAGATION MECHANISM COMPARISON (OPTIONAL)
# ====================================================================
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("CELL 13 — MECHANISM COMPARISON (subset of receivers)")
print("=" * 70)

SUBSET_SIZE = 20
rx_subset = receivers[:SUBSET_SIZE]
print(f"Using first {SUBSET_SIZE} receivers for comparison")

base_conf = {
    'max_depth': 8,
    'num_samples': 10_000_000,
    'seed': 42,
    'los': True,
    'reflection': True,
    'scattering': True,
    'diffraction': True,
}

scenarios = {
    'full': base_conf.copy(),
    'no_diffraction': {**base_conf, 'diffraction': False},
    'no_scattering': {**base_conf, 'scattering': False},
    'only_los_specular': {**base_conf, 'diffraction': False, 'scattering': False},
}

results = {}
for name, conf in scenarios.items():
    print(f"\n--- Running: {name} ---")
    for r in list(scene.receivers.keys()):
        scene.remove(r)
    for rx in rx_subset:
        scene.add(rx)
    try:
        t0 = time.time()
        paths = scene.compute_paths(**conf)
        elapsed = time.time() - t0
        a = extract_amplitudes(paths)
        a = np.squeeze(a)
        if a.ndim == 1:
            a = a[np.newaxis, :]
        power_lin = np.sum(np.abs(a)**2, axis=1)
        pl_db = -10 * np.log10(power_lin + 1e-15)
        results[name] = {'path_loss_db': pl_db, 'time_s': elapsed, 'num_receivers': len(rx_subset)}
        print(f"  Completed in {elapsed:.1f}s, mean PL = {np.mean(pl_db):.1f} dB")
    except Exception as e:
        print(f"  Failed: {e}")
        results[name] = None

for r in list(scene.receivers.keys()):
    scene.remove(r)
for rx in receivers:
    scene.add(rx)

os.makedirs(OUT_DIR, exist_ok=True)
comparison = []
for name, res in results.items():
    if res is not None:
        comparison.append({
            'scenario': name,
            'mean_path_loss_db': np.mean(res['path_loss_db']),
            'std_path_loss_db': np.std(res['path_loss_db']),
            'min_path_loss_db': np.min(res['path_loss_db']),
            'max_path_loss_db': np.max(res['path_loss_db']),
            'time_seconds': res['time_s'],
        })
df_comp = pd.DataFrame(comparison)
comp_csv = os.path.join(OUT_DIR, "propagation_mechanism_comparison.csv")
df_comp.to_csv(comp_csv, index=False)
print(f"\nSaved mechanism comparison to {comp_csv}")
print(df_comp.to_string())

plt.figure(figsize=(10,6))
plt.bar(df_comp['scenario'], df_comp['mean_path_loss_db'],
        yerr=df_comp['std_path_loss_db'], capsize=5, color='steelblue')
plt.ylabel('Mean Path Loss (dB)')
plt.title('Impact of Propagation Mechanisms on Path Loss')
plt.xticks(rotation=45, ha='right'); plt.grid(axis='y', alpha=0.3)
out_bar = os.path.join(OUT_DIR, "mechanism_comparison_bar.png")
plt.tight_layout(); plt.savefig(out_bar, dpi=150); plt.close()
print(f"Saved bar plot to {out_bar}")
print("\n Cell 13 complete")

## CELL 14 · Comprehensive Metrics for All Path Loss Types + Distance Analysis

In [ ]:
# ====================================================================
# CELL 14 — COMPREHENSIVE METRICS FOR ALL PATH LOSS TYPES + DISTANCE ANALYSIS
# ====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob, os
from scipy.stats import spearmanr

print("=" * 70)
print("CELL 14 — METRICS FOR BEST, INCOHERENT, COHERENT PATH LOSS")
print("=" * 70)
print(f"Using output directory: {OUT_DIR}")

summary_files = glob.glob(os.path.join(OUT_DIR, "path_solver_summary_*.csv"))
if not summary_files:
    raise FileNotFoundError(f"No PathSolver summary CSV found in {OUT_DIR}. Run Cell 9b first.")
sim_csv = max(summary_files, key=os.path.getctime)
df_sim = pd.read_csv(sim_csv)

required_cols = ['receiver', 'path_loss_best_db', 'path_loss_incoherent_db',
                 'path_loss_coherent_db', 'dist_from_tx_m', 'num_paths']
for col in required_cols:
    if col not in df_sim.columns:
        print(f"Warning: Column '{col}' missing.")
        if col == 'dist_from_tx_m':
            raise KeyError("Need 'dist_from_tx_m' for binning.")
df_sim = df_sim.dropna(subset=['path_loss_incoherent_db']).copy()

df_meas = pd.read_csv(MEASUREMENT_CSV)
if 'path_loss_dB' in df_meas.columns:
    meas_pl_col = 'path_loss_dB'
else:
    df_meas['path_loss_dB'] = EIRP_DBM - df_meas['local_measurement_dBm']
    meas_pl_col = 'path_loss_dB'
df_meas = df_meas[['name', meas_pl_col]].dropna()
df_meas.columns = ['receiver', 'measured_pl_db']

df_merged = pd.merge(df_sim, df_meas, on='receiver', how='inner')
print(f"Matched {len(df_merged)} receivers")

def compute_metrics(sim_pl, meas_pl):
    sim = np.array(sim_pl); meas = np.array(meas_pl)
    rmse = np.sqrt(np.mean((sim - meas)**2))
    mae = np.mean(np.abs(sim - meas))
    mse = np.mean((sim - meas)**2)
    ss_res = np.sum((sim - meas)**2)
    ss_tot = np.sum((meas - np.mean(meas))**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    bias = np.mean(sim - meas)
    try:
        rho, p = spearmanr(meas, sim)
    except:
        rho, p = np.nan, np.nan
    return {'RMSE': rmse, 'MAE': mae, 'MSE': mse, 'R2': r2, 'Bias': bias,
            'Spearman_rho': rho, 'Spearman_p': p}

types = {'best': 'path_loss_best_db', 'incoherent': 'path_loss_incoherent_db',
         'coherent': 'path_loss_coherent_db'}
metrics_all = {}
for name, col in types.items():
    if col not in df_merged.columns:
        print(f"Skipping {name}: column '{col}' missing"); continue
    sim_vals = df_merged[col].values
    meas_vals = df_merged['measured_pl_db'].values
    valid = ~np.isnan(sim_vals)
    metrics_all[name] = compute_metrics(sim_vals[valid], meas_vals[valid])

print("\n" + "=" * 80)
print("OVERALL METRICS (ALL RECEIVERS)")
print("=" * 80)
print(f"{'Type':<12} {'RMSE':>8} {'MAE':>8} {'MSE':>8} {'R2':>7} {'Bias':>8} {'Spearman':>10}")
for name, m in metrics_all.items():
    print(f"{name:<12} {m['RMSE']:>8.2f} {m['MAE']:>8.2f} {m['MSE']:>8.2f} {m['R2']:>7.3f} {m['Bias']:>8.2f} {m['Spearman_rho']:>10.3f}")

bin_width = 250
max_dist = df_merged['dist_from_tx_m'].max()
bins = np.arange(0, max_dist + bin_width, bin_width)
labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]
df_merged['dist_bin'] = pd.cut(df_merged['dist_from_tx_m'], bins=bins, labels=labels)

bin_stats = []
for bin_label in labels:
    bin_data = df_merged[df_merged['dist_bin'] == bin_label]
    if len(bin_data) == 0: continue
    row = {'dist_bin': bin_label, 'avg_dist_m': bin_data['dist_from_tx_m'].mean(),
           'n_receivers': len(bin_data), 'avg_num_paths': bin_data['num_paths'].mean()}
    for name, col in types.items():
        if col not in bin_data.columns: continue
        sim_vals = bin_data[col].values; meas_vals = bin_data['measured_pl_db'].values
        valid = ~np.isnan(sim_vals)
        if np.sum(valid) < 3:
            row[f'{name}_RMSE'] = np.nan; row[f'{name}_bias'] = np.nan
        else:
            row[f'{name}_RMSE'] = np.sqrt(np.mean((sim_vals[valid]-meas_vals[valid])**2))
            row[f'{name}_bias'] = np.mean(sim_vals[valid]-meas_vals[valid])
    bin_stats.append(row)
df_bin = pd.DataFrame(bin_stats)
print("\nSLIDING BIN ANALYSIS:"); print(df_bin.round(2).to_string(index=False))
df_bin.to_csv(os.path.join(OUT_DIR, "distance_bin_metrics.csv"), index=False)

cumulative_edges = np.arange(250, max_dist + 250, 250)
cumulative_results = []
for max_d in cumulative_edges:
    bin_data = df_merged[df_merged['dist_from_tx_m'] <= max_d]
    if len(bin_data) < 3: continue
    row = {'max_dist_m': max_d, 'avg_dist_m': bin_data['dist_from_tx_m'].mean(),
           'n_receivers': len(bin_data), 'avg_num_paths': bin_data['num_paths'].mean()}
    for name, col in types.items():
        if col not in bin_data.columns: continue
        sim_vals = bin_data[col].values; meas_vals = bin_data['measured_pl_db'].values
        valid = ~np.isnan(sim_vals)
        if np.sum(valid) < 3:
            for k in ['RMSE','MAE','MSE','R2','bias']: row[f'{name}_{k}'] = np.nan
        else:
            m = compute_metrics(sim_vals[valid], meas_vals[valid])
            row[f'{name}_RMSE'] = m['RMSE']; row[f'{name}_MAE'] = m['MAE']
            row[f'{name}_MSE'] = m['MSE']; row[f'{name}_R2'] = m['R2']
            row[f'{name}_bias'] = m['Bias']
    cumulative_results.append(row)
df_cumulative = pd.DataFrame(cumulative_results)
df_cumulative.to_csv(os.path.join(OUT_DIR, "cumulative_path_loss_metrics.csv"), index=False)
print(f"Saved cumulative metrics to {os.path.join(OUT_DIR, 'cumulative_path_loss_metrics.csv')}")

fig_cum, axes_cum = plt.subplots(2, 2, figsize=(14, 10))
for idx, metric in enumerate(['RMSE','MAE','R2','bias']):
    ax = axes_cum[idx//2, idx%2]
    for name in types.keys():
        col = f'{name}_{metric}'
        if col in df_cumulative.columns:
            ax.plot(df_cumulative['max_dist_m'], df_cumulative[col], 'o-', label=name)
    ax.set_xlabel('Max Distance in Bin (m)'); ax.set_ylabel(metric)
    ax.set_title(f'Cumulative {metric} vs Distance'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cumulative_metrics_plot.png"), dpi=150); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for name, col in types.items():
    if f'{name}_RMSE' in df_bin.columns:
        axes[0,0].plot(df_bin['avg_dist_m'], df_bin[f'{name}_RMSE'], 'o-', label=name)
    if f'{name}_bias' in df_bin.columns:
        axes[0,1].plot(df_bin['avg_dist_m'], df_bin[f'{name}_bias'], 'o-', label=name)
axes[0,0].set(xlabel='Distance (m)', ylabel='RMSE (dB)', title='RMSE vs Distance'); axes[0,0].legend(); axes[0,0].grid(True)
axes[0,1].axhline(0,color='k',linestyle='--'); axes[0,1].set(xlabel='Distance (m)', ylabel='Bias [dB]', title='Bias vs Distance'); axes[0,1].legend(); axes[0,1].grid(True)
axes[1,0].bar(df_bin['avg_dist_m'], df_bin['avg_num_paths'], width=150, alpha=0.7)
axes[1,0].set(xlabel='Distance (m)', ylabel='Avg Paths', title='Multipath Count'); axes[1,0].grid(True)
axes[1,1].bar(df_bin['avg_dist_m'], df_bin['n_receivers'], width=150, alpha=0.7, color='orange')
axes[1,1].set(xlabel='Distance (m)', ylabel='N Receivers', title='Sample Size per Bin'); axes[1,1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "distance_analysis.png"), dpi=150); plt.show()

if 'num_paths' in df_merged.columns:
    fig, ax = plt.subplots(figsize=(8,5))
    ax.scatter(df_merged['num_paths'], df_merged['path_loss_incoherent_db']-df_merged['measured_pl_db'], alpha=0.3, s=5)
    ax.set(xlabel='Number of Paths', ylabel='Error (Incoherent-Measured) [dB]', title='Error vs. Number of Paths'); ax.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "error_vs_num_paths.png"), dpi=150); plt.show()

print("\n Cell 14 complete")

## CELL 14b · Cumulative Distance Bin Metrics: Empirical Models + Sionna

In [ ]:
# ====================================================================
# CELL 14b — CUMULATIVE DISTANCE BIN METRICS: EMPIRICAL MODELS + SIONNA
# ====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from scipy.constants import c
print("=" * 70); print("CELL 14b — CUMULATIVE BINS: EMPIRICAL MODELS vs SIONNA"); print("=" * 70)
print(f"Using output directory: {OUT_DIR}")
if 'df_merged' not in locals(): raise NameError("df_merged not found. Run CELL 14 first.")
print(f"Using existing df_merged with {len(df_merged)} matched receivers.")
for col in ['path_loss_best_db','path_loss_incoherent_db','path_loss_coherent_db']:
    if col not in df_merged.columns: raise KeyError(f"Column '{col}' missing. Re-run CELL 9b and CELL 14.")

FREQ_HZ = FREQUENCY_HZ
LAMBDA = c / FREQ_HZ

def free_space_pl(d):
    d = np.maximum(d, 1e-3)
    return 20*np.log10(d) + 20*np.log10(FREQ_HZ) - 147.55

def two_ray_pl(d, ht=TX_AGL_M, hr=1.5):
    d = np.maximum(d, 1e-3)
    d_break = 4*ht*hr/LAMBDA
    return np.where(d<=d_break, free_space_pl(d), 40*np.log10(d)-20*np.log10(ht)-20*np.log10(hr))

def log_distance_pl(d, pl0=40.0, n=3.5, d0=1.0):
    d = np.maximum(d, d0)
    return pl0 + 10*n*np.log10(d/d0)

df_merged['pl_free_space'] = free_space_pl(df_merged['dist_from_tx_m'])
df_merged['pl_two_ray'] = two_ray_pl(df_merged['dist_from_tx_m'])
df_merged['pl_log_distance'] = log_distance_pl(df_merged['dist_from_tx_m'])

def compute_metrics(sim, meas):
    sim=np.array(sim); meas=np.array(meas)
    valid=~np.isnan(sim)&~np.isnan(meas); sim=sim[valid]; meas=meas[valid]
    if len(sim)<2: return {'RMSE':np.nan,'MAE':np.nan,'MSE':np.nan,'R2':np.nan,'Bias':np.nan}
    rmse=np.sqrt(np.mean((sim-meas)**2)); mae=np.mean(np.abs(sim-meas)); mse=np.mean((sim-meas)**2)
    ss_res=np.sum((sim-meas)**2); ss_tot=np.sum((meas-np.mean(meas))**2)
    r2=1-(ss_res/ss_tot) if ss_tot!=0 else 0; bias=np.mean(sim-meas)
    return {'RMSE':rmse,'MAE':mae,'MSE':mse,'R2':r2,'Bias':bias}

models = {'Sionna Best':'path_loss_best_db','Sionna Incoherent':'path_loss_incoherent_db',
          'Sionna Coherent':'path_loss_coherent_db','Free Space':'pl_free_space',
          'Two-Ray':'pl_two_ray','Log-Distance':'pl_log_distance'}

max_dist = df_merged['dist_from_tx_m'].max()
cumulative_edges = np.arange(250, max_dist+250, 250)
cumulative_results = []
for max_d in cumulative_edges:
    bin_data = df_merged[df_merged['dist_from_tx_m']<=max_d]
    if len(bin_data)<3: continue
    row = {'max_dist_m':max_d,'avg_dist_m':bin_data['dist_from_tx_m'].mean(),'n_receivers':len(bin_data)}
    for name, col in models.items():
        m = compute_metrics(bin_data[col].values, bin_data['measured_pl_db'].values)
        for k,v in m.items(): row[f'{name}_{k}'] = v
    cumulative_results.append(row)
df_cum = pd.DataFrame(cumulative_results)
print("\nCumulative metrics (first 10 rows):"); print(df_cum.head(10).round(2).to_string(index=False))
cum_csv = os.path.join(OUT_DIR,"cumulative_metrics_all_models.csv"); df_cum.to_csv(cum_csv,index=False)
print(f"Saved cumulative metrics to {cum_csv}")

fig,axes=plt.subplots(2,2,figsize=(16,10))
for idx,metric in enumerate(['RMSE','MAE','R2','Bias']):
    ax=axes[idx//2,idx%2]
    for name in models.keys():
        col=f'{name}_{metric}'
        if col in df_cum.columns: ax.plot(df_cum['max_dist_m'],df_cum[col],'o-',label=name)
    ax.set_xlabel('Max Distance in Bin (m)'); ax.set_ylabel(metric)
    ax.set_title(f'Cumulative {metric}'); ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
out_plot=os.path.join(OUT_DIR,"cumulative_metrics_all_models.png")
plt.savefig(out_plot,dpi=150); plt.show()
print(f"Saved plot to {out_plot}")
print("\n CELL 14b complete")

## CELL 14c · Cumulative Distance Bin Metrics (0-250, 0-500, …) for All Models

In [ ]:
# ====================================================================
# CELL 14c — CUMULATIVE DISTANCE BIN METRICS (0‑250, 0‑500, …) FOR ALL MODELS
# ====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from scipy.constants import c
print("=" * 70); print("CELL 14c — CUMULATIVE BINS (0‑250, 0‑500, …): EMPIRICAL vs SIONNA"); print("=" * 70)
print(f"Using output directory: {OUT_DIR}")
if 'df_merged' not in locals(): raise NameError("df_merged not found. Run CELL 14 first.")
print(f"Using existing df_merged with {len(df_merged)} matched receivers.")
for col in ['path_loss_best_db','path_loss_incoherent_db','path_loss_coherent_db']:
    if col not in df_merged.columns: raise KeyError(f"Sionna column '{col}' missing. Re-run CELL 9b and CELL 14.")

FREQ_HZ = FREQUENCY_HZ; LAMBDA = c/FREQ_HZ

def free_space_pl(d):
    d=np.maximum(d,1e-3); return 20*np.log10(d)+20*np.log10(FREQ_HZ)-147.55

def two_ray_pl(d, ht=TX_AGL_M, hr=1.5):
    d=np.maximum(d,1e-3); d_break=4*ht*hr/LAMBDA
    return np.where(d<=d_break, free_space_pl(d), 40*np.log10(d)-20*np.log10(ht)-20*np.log10(hr))

def log_distance_pl(d, pl0=40.0, n=3.5, d0=1.0):
    d=np.maximum(d,d0); return pl0+10*n*np.log10(d/d0)

df_merged['pl_free_space'] = free_space_pl(df_merged['dist_from_tx_m'])
df_merged['pl_two_ray'] = two_ray_pl(df_merged['dist_from_tx_m'])
df_merged['pl_log_distance'] = log_distance_pl(df_merged['dist_from_tx_m'])

def compute_metrics(sim, meas):
    sim=np.array(sim); meas=np.array(meas); valid=~np.isnan(sim)&~np.isnan(meas)
    sim=sim[valid]; meas=meas[valid]
    if len(sim)<2: return {'RMSE':np.nan,'MAE':np.nan,'MSE':np.nan,'R2':np.nan,'Bias':np.nan}
    rmse=np.sqrt(np.mean((sim-meas)**2)); mae=np.mean(np.abs(sim-meas)); mse=np.mean((sim-meas)**2)
    ss_res=np.sum((sim-meas)**2); ss_tot=np.sum((meas-np.mean(meas))**2)
    r2=1-(ss_res/ss_tot) if ss_tot!=0 else 0; bias=np.mean(sim-meas)
    return {'RMSE':rmse,'MAE':mae,'MSE':mse,'R2':r2,'Bias':bias}

models = {'Sionna Best':'path_loss_best_db','Sionna Incoherent':'path_loss_incoherent_db',
          'Sionna Coherent':'path_loss_coherent_db','Free Space':'pl_free_space',
          'Two-Ray':'pl_two_ray','Log-Distance':'pl_log_distance'}

max_dist=df_merged['dist_from_tx_m'].max()
cumulative_edges=np.arange(250,max_dist+250,250)
cumulative_results=[]
for max_d in cumulative_edges:
    bin_data=df_merged[df_merged['dist_from_tx_m']<=max_d]
    if len(bin_data)<3: continue
    row={'max_dist_m':max_d,'avg_dist_m':bin_data['dist_from_tx_m'].mean(),'n_receivers':len(bin_data)}
    for name,col in models.items():
        m=compute_metrics(bin_data[col].values, bin_data['measured_pl_db'].values)
        for k,v in m.items(): row[f'{name}_{k}']=v
    cumulative_results.append(row)
df_cum=pd.DataFrame(cumulative_results)
print("\nCumulative bin metrics (all models, 0‑250, 0‑500, …):")
print(df_cum.round(2).to_string(index=False))
cum_csv=os.path.join(OUT_DIR,"cumulative_metrics_all_models.csv")
df_cum.to_csv(cum_csv,index=False); print(f"Saved to {cum_csv}")

fig,axes=plt.subplots(2,2,figsize=(16,10))
for idx,metric in enumerate(['RMSE','MAE','R2','Bias']):
    ax=axes[idx//2,idx%2]
    for name in models.keys():
        col=f'{name}_{metric}'
        if col in df_cum.columns: ax.plot(df_cum['max_dist_m'],df_cum[col],'o-',label=name)
    ax.set_xlabel('Max Distance in Bin (m)'); ax.set_ylabel(metric)
    ax.set_title(f'Cumulative {metric} vs Distance'); ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
out_plot=os.path.join(OUT_DIR,"cumulative_metrics_all_models.png")
plt.savefig(out_plot,dpi=150); plt.show(); print(f"Saved plot to {out_plot}")
print("\n CELL 14c complete")

## CELL 14e · Research-Guided Per-Ray Metrics

In [ ]:
# ====================================================================
# CELL 14e — RESEARCH-GUIDED PER‑RAY METRICS
# ====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import os, glob
from scipy import stats
from scipy.constants import speed_of_light as c

print("=" * 70); print("CELL 14e — PER‑RAY CHANNEL METRICS (Literature-Based)"); print("=" * 70)
print(f"Using output directory: {OUT_DIR}")

per_ray_files = glob.glob(os.path.join(OUT_DIR, "path_solver_per_ray_*.csv"))
if not per_ray_files:
    if 'paths' in globals() and paths is not None:
        print("No per‑ray CSV found. Creating from 'paths' object...")
        a_data = extract_amplitudes(paths)
        tau_numpy = paths.tau.numpy() if hasattr(paths.tau,'numpy') else np.array(paths.tau)
        ray_list = []
        for rx_idx in range(a_data.shape[0]):
            for p_idx in range(a_data.shape[1]):
                if np.isnan(a_data[rx_idx, p_idx]): continue
                power_linear = np.abs(a_data[rx_idx, p_idx])**2
                ray_list.append({'receiver_name':f'receiver_{rx_idx}','ray_idx':p_idx,
                    'power_linear':power_linear,'delay_s':tau_numpy[rx_idx,p_idx],
                    'path_loss_db':-10*np.log10(power_linear+1e-12),
                    'distance_m':tau_numpy[rx_idx,p_idx]*c})
        df_ray = pd.DataFrame(ray_list)
        temp_csv = os.path.join(OUT_DIR,"path_solver_per_ray_temp.csv")
        df_ray.to_csv(temp_csv,index=False)
        print(f"Created temporary per‑ray CSV with {len(df_ray)} rays at {temp_csv}")
    else:
        raise FileNotFoundError(f"No per‑ray CSV in {OUT_DIR} and no 'paths' object. Run Cell 9b first.")
else:
    per_ray_csv = max(per_ray_files, key=os.path.getctime)
    df_ray = pd.read_csv(per_ray_csv)
    print(f"Loaded {len(df_ray)} rays from {os.path.basename(per_ray_csv)}")

if 'los' not in df_ray.columns:
    print("Warning: 'los' column missing. Using distance threshold (500 m).")
    if 'receiver_name' not in df_ray.columns:
        raise KeyError("No receiver identifiers to assign LOS/NLOS.")
    summary_files = glob.glob(os.path.join(OUT_DIR,"path_solver_summary_*.csv"))
    if summary_files:
        df_sum = pd.read_csv(max(summary_files,key=os.path.getctime))
        if 'receiver' not in df_sum.columns: raise KeyError("Summary CSV missing 'receiver' column.")
        dist_map = df_sum.set_index('receiver')['dist_from_tx_m'].to_dict()
        df_ray['dist_m'] = df_ray['receiver_name'].map(dist_map)
        df_ray['los'] = df_ray['dist_m'] < 500
        df_ray = df_ray.dropna(subset=['los'])
    else:
        raise FileNotFoundError("Need summary CSV for distance-based LOS classification.")
print(f"LOS rays: {df_ray['los'].sum()} | NLOS rays: {(~df_ray['los']).sum()}")

def compute_rms_delay_spread(df, rx_name):
    rx_df=df[df['receiver_name']==rx_name]; power=rx_df['power_linear'].values
    delays=rx_df['delay_s'].values*1e9
    if len(power)<2 or np.sum(power)==0: return np.nan
    mean_delay=np.sum(power*delays)/np.sum(power)
    return np.sqrt(np.sum(power*(delays-mean_delay)**2)/np.sum(power))

def compute_k_factor(df, rx_name):
    rx_df=df[df['receiver_name']==rx_name]; power=rx_df['power_linear'].values
    if len(power)<2: return np.nan
    p_max=np.max(power); p_diffuse=np.sum(power)-p_max
    if p_diffuse<=0: return np.inf
    return 10*np.log10(p_max/p_diffuse)

def compute_constructive_fraction(df, rx_name):
    rx_df=df[df['receiver_name']==rx_name]
    if 'constructive_destructive' not in rx_df.columns: return np.nan
    total=len(rx_df)
    if total==0: return np.nan
    return (rx_df['constructive_destructive']=='CONSTRUCTIVE').sum()/total

rx_names = df_ray['receiver_name'].unique()
metrics_data = []
for rx in rx_names:
    rx_df=df_ray[df_ray['receiver_name']==rx]
    if len(rx_df)==0: continue
    metrics_data.append({
        'receiver':rx,
        'dist_from_tx_m':rx_df['dist_m'].iloc[0] if 'dist_m' in rx_df.columns else np.nan,
        'los':rx_df['los'].iloc[0] if 'los' in rx_df.columns else None,
        'num_paths':len(rx_df),
        'num_constructive':(rx_df['constructive_destructive']=='CONSTRUCTIVE').sum() if 'constructive_destructive' in rx_df.columns else 0,
        'constructive_fraction':compute_constructive_fraction(df_ray,rx),
        'rms_delay_spread_ns':compute_rms_delay_spread(df_ray,rx),
        'k_factor_db':compute_k_factor(df_ray,rx),
        'dominant_ray_type':rx_df['ray_type'].mode().iloc[0] if 'ray_type' in rx_df.columns and len(rx_df)>0 else 'UNKNOWN'
    })
df_metrics = pd.DataFrame(metrics_data)

metrics_csv = os.path.join(OUT_DIR,"per_receiver_metrics.csv")
df_metrics.to_csv(metrics_csv,index=False)
print(f"\nSaved per‑receiver metrics to {metrics_csv}")
print(df_metrics.describe())

sns.set_style("whitegrid")
fig,axes = plt.subplots(2,3,figsize=(16,10))
for los_val,label,color in [(True,'LOS','blue'),(False,'NLOS','red')]:
    subset=df_metrics[df_metrics['los']==los_val]['rms_delay_spread_ns'].dropna()
    if len(subset)>0: axes[0,0].hist(subset,bins=20,alpha=0.5,color=color,label=label,density=True)
axes[0,0].set(xlabel='RMS Delay Spread (ns)',ylabel='Density',title='(a) RMS Delay Spread'); axes[0,0].legend(); axes[0,0].grid(True,alpha=0.3)

for los_val,label,color in [(True,'LOS','blue'),(False,'NLOS','red')]:
    subset=df_metrics[df_metrics['los']==los_val]['k_factor_db'].dropna()
    subset=subset[np.isfinite(subset)]
    if len(subset)>0: axes[0,1].hist(subset,bins=20,alpha=0.5,color=color,label=label,density=True)
axes[0,1].set(xlabel='K‑factor (dB)',ylabel='Density',title='(b) Rician K‑Factor'); axes[0,1].legend(); axes[0,1].grid(True,alpha=0.3)

for los_val,label,color in [(True,'LOS','blue'),(False,'NLOS','red')]:
    subset=df_metrics[df_metrics['los']==los_val]
    axes[0,2].scatter(subset['num_paths'],subset['constructive_fraction'],alpha=0.5,c=color,label=label,s=20)
axes[0,2].set(xlabel='Number of Paths',ylabel='Constructive Fraction',title='(c) Constructive Fraction'); axes[0,2].legend(); axes[0,2].grid(True,alpha=0.3)

if 'dist_from_tx_m' in df_metrics.columns and df_metrics['dist_from_tx_m'].notna().any():
    for los_val,label,color in [(True,'LOS','blue'),(False,'NLOS','red')]:
        subset=df_metrics[df_metrics['los']==los_val].dropna(subset=['dist_from_tx_m','rms_delay_spread_ns'])
        if len(subset)>0: axes[1,0].scatter(subset['dist_from_tx_m'],subset['rms_delay_spread_ns'],alpha=0.5,c=color,label=label,s=20)
    axes[1,0].set(xlabel='Distance from TX (m)',ylabel='RMS Delay Spread (ns)',title='(d) Delay Spread vs Distance'); axes[1,0].legend(); axes[1,0].grid(True,alpha=0.3)
else:
    axes[1,0].text(0.5,0.5,'Distance data not available',ha='center',va='center'); axes[1,0].set_title('(d) unavailable')

if 'dist_from_tx_m' in df_metrics.columns and df_metrics['dist_from_tx_m'].notna().any():
    for los_val,label,color in [(True,'LOS','blue'),(False,'NLOS','red')]:
        subset=df_metrics[df_metrics['los']==los_val].copy()
        subset['k_factor_db']=subset['k_factor_db'].replace([np.inf,-np.inf],np.nan)
        subset=subset.dropna(subset=['dist_from_tx_m','k_factor_db'])
        if len(subset)>0: axes[1,1].scatter(subset['dist_from_tx_m'],subset['k_factor_db'],alpha=0.5,c=color,label=label,s=20)
    axes[1,1].set(xlabel='Distance from TX (m)',ylabel='K‑factor (dB)',title='(e) K‑factor vs Distance'); axes[1,1].legend(); axes[1,1].grid(True,alpha=0.3)
else:
    axes[1,1].text(0.5,0.5,'Distance data not available',ha='center',va='center'); axes[1,1].set_title('(e) unavailable')

if 'ray_type' in df_ray.columns:
    ray_type_counts=df_ray['ray_type'].value_counts()
    if len(ray_type_counts)>0: axes[1,2].pie(ray_type_counts.values,labels=ray_type_counts.index,autopct='%1.1f%%')
    axes[1,2].set_title('(f) Ray Type Composition')
else:
    axes[1,2].text(0.5,0.5,'Ray type data not available',ha='center',va='center'); axes[1,2].set_title('(f) unavailable')

plt.tight_layout()
out_fig=os.path.join(OUT_DIR,"per_ray_channel_metrics.png"); plt.savefig(out_fig,dpi=150); plt.show()
print(f"Saved plots to {out_fig}")

print("\n" + "=" * 80); print("STATISTICAL SUMMARY (cf. Literature)"); print("=" * 80)
for los_label in [True,False]:
    subset=df_metrics[df_metrics['los']==los_label]
    print(f"\n{'LOS' if los_label else 'NLOS'} Receivers (n={len(subset)}):")
    rms_vals=subset['rms_delay_spread_ns'].dropna()
    if len(rms_vals)>0: print(f"  RMS Delay Spread: mean={rms_vals.mean():.2f} ns, std={rms_vals.std():.2f} ns")
    k_vals=subset['k_factor_db'].replace([np.inf,-np.inf],np.nan).dropna()
    if len(k_vals)>0: print(f"  K‑factor: mean={k_vals.mean():.2f} dB, std={k_vals.std():.2f} dB")
    cf_vals=subset['constructive_fraction'].dropna()
    if len(cf_vals)>0: print(f"  Constructive fraction: mean={cf_vals.mean():.3f}, std={cf_vals.std():.3f}")
print("\n Cell 14e complete")

## CELL 14f · Poisson Goodness-of-Fit Test for LOS RMS Delay Spread

In [ ]:
# ====================================================================
# CELL 14f — POISSON GOODNESS‑OF‑FIT TEST FOR LOS RMS DELAY SPREAD
# ====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from scipy import stats

print("=" * 70); print("CELL 14f — POISSON GOODNESS‑OF‑FIT (LOS RMS Delay Spread)"); print("=" * 70)
print(f"Using output directory: {OUT_DIR}")

metrics_file = os.path.join(OUT_DIR,"per_receiver_metrics.csv")
if not os.path.exists(metrics_file):
    raise FileNotFoundError(f"per_receiver_metrics.csv not found in {OUT_DIR}. Run CELL 14e first.")
df_metrics = pd.read_csv(metrics_file)
print(f"Loaded {len(df_metrics)} receivers from {metrics_file}")

los_data = df_metrics[df_metrics['los']==True]['rms_delay_spread_ns'].dropna()
if len(los_data)==0: raise ValueError("No LOS receivers with valid RMS delay spread found.")
print(f"LOS receivers with valid RMS delay spread: {len(los_data)}")
print(f"Data range: {los_data.min():.2f} – {los_data.max():.2f} ns")

lambda_poisson = los_data.mean()
print(f"\nEstimated Poisson parameter lambda = {lambda_poisson:.3f} ns")

data_int = np.round(los_data).astype(int)
value_counts = pd.Series(data_int).value_counts().sort_index()
observed_counts = value_counts.values; observed_bins = value_counts.index.values

if len(observed_bins)>1:
    bin_edges = np.append(observed_bins, observed_bins[-1]+1)
else:
    bin_edges = np.array([observed_bins[0], observed_bins[0]+1])

expected_counts = len(los_data) * (
    stats.poisson.cdf(bin_edges[1:], lambda_poisson) -
    stats.poisson.cdf(bin_edges[:-1], lambda_poisson))

min_expected=1.0; merged_obs=[]; merged_exp=[]; cum_obs=0; cum_exp=0
for obs,exp in zip(observed_counts, expected_counts):
    cum_obs+=obs; cum_exp+=exp
    if cum_exp>=min_expected:
        merged_obs.append(cum_obs); merged_exp.append(cum_exp); cum_obs=0; cum_exp=0
if cum_obs>0 or cum_exp>0:
    merged_obs.append(cum_obs); merged_exp.append(cum_exp)

observed_counts_final=np.array(merged_obs); expected_counts_final=np.array(merged_exp)
print(f"\nSum of observed frequencies: {observed_counts_final.sum()}")
print(f"Sum of expected frequencies: {expected_counts_final.sum():.2f} (original N={len(los_data)})")

if len(observed_counts_final)<2:
    print("Not enough bins after merging. Test cannot be performed.")
    chi2_stat=np.nan; p_value=np.nan
else:
    chi2_stat,p_value=stats.chisquare(f_obs=observed_counts_final, f_exp=expected_counts_final)

print("\n" + "=" * 80); print("GOODNESS‑OF‑FIT TEST: LOS RMS DELAY SPREAD vs POISSON"); print("=" * 80)
print(f"Chi‑square statistic: {chi2_stat:.3f}")
print(f"Degrees of freedom (approximate): {len(observed_counts_final)-1}")
print(f"p‑value: {p_value:.5f}")
print("\nInterpretation:")
if np.isnan(p_value):
    print("  Test could not be performed (insufficient data or binning issue).")
elif p_value>0.05:
    print("  p > 0.05 — Fail to reject null hypothesis.")
    print("     The LOS RMS delay spread follows a Poisson distribution.")
else:
    print("  p <= 0.05 — Reject null hypothesis.")
    print("     The LOS RMS delay spread does NOT follow a Poisson distribution.")
    print("     Consider tuning max_depth, material properties, or scene detail.")

fig,ax=plt.subplots(figsize=(10,6))
ax.hist(los_data,bins=20,density=True,alpha=0.5,color='blue',label='Observed (LOS)')
x_vals=np.arange(0,int(los_data.max())+1,1)
pmf_poisson=stats.poisson.pmf(x_vals,lambda_poisson)
ax.plot(x_vals,pmf_poisson,'ro-',label=f'Poisson (lambda={lambda_poisson:.2f})')
ax.set(xlabel='RMS Delay Spread (ns)',ylabel='Probability Density',
       title='Goodness‑of‑Fit: Poisson vs Observed LOS RMS Delay Spread')
ax.legend(); ax.grid(True,alpha=0.3)
out_plot=os.path.join(OUT_DIR,"poisson_fit_los_rms_delay.png")
plt.savefig(out_plot,dpi=150); plt.show()
print(f"\nSaved Poisson fit plot to {out_plot}")
print("\n CELL 14f complete")

## CELL 15 · Diagnostic – Error as Function of Distance

In [ ]:
#  CELL 15
#  Diagnostic: error as function of distance
import matplotlib.pyplot as plt

df_merged['error_db'] = df_merged['path_loss_incoherent_db'] - df_merged['measured_pl_db']
if 'dist_from_tx_m' in df_merged.columns:
    dist = df_merged['dist_from_tx_m']
else:
    tx_pos = np.array([_safe(tx.position[0]), _safe(tx.position[1]), _safe(tx.position[2])])
    rx_pos = df_merged[['x_m','y_m','z_m']].values
    dist = np.linalg.norm(rx_pos - tx_pos, axis=1)

plt.figure(figsize=(10,5))
plt.scatter(dist, df_merged['error_db'], alpha=0.5, s=5)
plt.axhline(0, color='k', linestyle='--')
plt.xlabel('Distance from TX (m)'); plt.ylabel('Error (Sim - Meas) [dB]')
plt.title('Prediction Error vs Distance'); plt.grid(True)
plt.show()

bins = [0, 500, 1000, 2000, 5000]
df_merged['dist_bin'] = pd.cut(dist, bins)
print(df_merged.groupby('dist_bin')['error_db'].describe())
plt.savefig(os.path.join(OUT_DIR, "error_vs_distance.png"), dpi=150)

## CELL 16 · Comprehensive Metrics for Best, Incoherent, Coherent Path Loss

In [ ]:
# ====================================================================
# CELL 16 — COMPREHENSIVE METRICS FOR BEST, INCOHERENT, COHERENT PATH LOSS
# ====================================================================
import pandas as pd, numpy as np, matplotlib.pyplot as plt, glob, os
from scipy.stats import spearmanr

print("=" * 70); print("CELL 16 — METRICS FOR BEST, INCOHERENT, COHERENT PATH LOSS"); print("=" * 70)

summary_files = glob.glob(os.path.join(OUT_DIR,"path_solver_summary_*.csv"))
if not summary_files: raise FileNotFoundError("No PathSolver summary CSV found. Run Cell 9b first.")
sim_csv = max(summary_files, key=os.path.getctime)
df_sim = pd.read_csv(sim_csv)
for col in ['receiver','path_loss_best_db','path_loss_incoherent_db','path_loss_coherent_db','dist_from_tx_m','num_paths']:
    if col not in df_sim.columns: print(f"Warning: Column '{col}' missing.")
df_sim = df_sim.dropna(subset=['path_loss_incoherent_db']).copy()

df_meas = pd.read_csv(MEASUREMENT_CSV)
if 'path_loss_dB' in df_meas.columns:
    meas_pl_col = 'path_loss_dB'
else:
    df_meas['path_loss_dB'] = EIRP_DBM - df_meas['local_measurement_dBm']
    meas_pl_col = 'path_loss_dB'
df_meas = df_meas[['name', meas_pl_col]].dropna()
df_meas.columns = ['receiver', 'measured_pl_db']

df_merged = pd.merge(df_sim, df_meas, on='receiver', how='inner')
print(f"Matched {len(df_merged)} receivers")

def compute_metrics(sim_pl, meas_pl):
    sim=np.array(sim_pl); meas=np.array(meas_pl)
    rmse=np.sqrt(np.mean((sim-meas)**2)); mae=np.mean(np.abs(sim-meas)); mse=np.mean((sim-meas)**2)
    ss_res=np.sum((sim-meas)**2); ss_tot=np.sum((meas-np.mean(meas))**2)
    r2=1-(ss_res/ss_tot) if ss_tot!=0 else 0; bias=np.mean(sim-meas)
    try: rho,p=spearmanr(meas,sim)
    except: rho,p=np.nan,np.nan
    return {'RMSE':rmse,'MAE':mae,'MSE':mse,'R2':r2,'Bias':bias,'Spearman_rho':rho,'Spearman_p':p}

types={'best':'path_loss_best_db','incoherent':'path_loss_incoherent_db','coherent':'path_loss_coherent_db'}
metrics_all={}
for name,col in types.items():
    if col not in df_merged.columns: continue
    sim_vals=df_merged[col].values; meas_vals=df_merged['measured_pl_db'].values
    valid=~np.isnan(sim_vals)
    metrics_all[name]=compute_metrics(sim_vals[valid], meas_vals[valid])

print("\n" + "=" * 80); print("OVERALL METRICS (ALL RECEIVERS)"); print("=" * 80)
print(f"{'Type':<12} {'RMSE':>8} {'MAE':>8} {'MSE':>8} {'R2':>7} {'Bias':>8} {'Spearman':>10}")
for name,m in metrics_all.items():
    print(f"{name:<12} {m['RMSE']:>8.2f} {m['MAE']:>8.2f} {m['MSE']:>8.2f} {m['R2']:>7.3f} {m['Bias']:>8.2f} {m['Spearman_rho']:>10.3f}")

bin_width=250; max_dist=df_merged['dist_from_tx_m'].max()
bins=np.arange(0,max_dist+bin_width,bin_width)
labels=[f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]
df_merged['dist_bin']=pd.cut(df_merged['dist_from_tx_m'],bins=bins,labels=labels)

bin_stats=[]
for bin_label in labels:
    bin_data=df_merged[df_merged['dist_bin']==bin_label]
    if len(bin_data)==0: continue
    row={'dist_bin':bin_label,'avg_dist_m':bin_data['dist_from_tx_m'].mean(),
         'n_receivers':len(bin_data),'avg_num_paths':bin_data['num_paths'].mean()}
    for name,col in types.items():
        if col not in bin_data.columns: continue
        sv=bin_data[col].values; mv=bin_data['measured_pl_db'].values; valid=~np.isnan(sv)
        if np.sum(valid)<3: row[f'{name}_RMSE']=np.nan; row[f'{name}_bias']=np.nan
        else:
            row[f'{name}_RMSE']=np.sqrt(np.mean((sv[valid]-mv[valid])**2))
            row[f'{name}_bias']=np.mean(sv[valid]-mv[valid])
    bin_stats.append(row)
df_bin=pd.DataFrame(bin_stats)
print(df_bin.round(2).to_string(index=False))
df_bin.to_csv(os.path.join(OUT_DIR,"distance_bin_metrics.csv"),index=False)
print(f"Saved distance bin metrics")

fig,axes=plt.subplots(2,2,figsize=(14,10))
for name,col in types.items():
    if f'{name}_RMSE' in df_bin.columns: axes[0,0].plot(df_bin['avg_dist_m'],df_bin[f'{name}_RMSE'],'o-',label=name)
    if f'{name}_bias' in df_bin.columns: axes[0,1].plot(df_bin['avg_dist_m'],df_bin[f'{name}_bias'],'o-',label=name)
axes[0,0].set(xlabel='Distance (m)',ylabel='RMSE (dB)',title='RMSE vs Distance'); axes[0,0].legend(); axes[0,0].grid(True)
axes[0,1].axhline(0,color='k',linestyle='--'); axes[0,1].set(xlabel='Distance (m)',ylabel='Bias [dB]',title='Bias vs Distance'); axes[0,1].legend(); axes[0,1].grid(True)
axes[1,0].bar(df_bin['avg_dist_m'],df_bin['avg_num_paths'],width=150,alpha=0.7)
axes[1,0].set(xlabel='Distance (m)',ylabel='Avg Number of Paths',title='Multipath Count'); axes[1,0].grid(True)
axes[1,1].bar(df_bin['avg_dist_m'],df_bin['n_receivers'],width=150,alpha=0.7,color='orange')
axes[1,1].set(xlabel='Distance (m)',ylabel='Number of Receivers',title='Sample Size per Bin'); axes[1,1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"distance_analysis.png"),dpi=150); plt.show()

if 'num_paths' in df_merged.columns:
    fig,ax=plt.subplots(figsize=(8,5))
    ax.scatter(df_merged['num_paths'],df_merged['path_loss_incoherent_db']-df_merged['measured_pl_db'],alpha=0.3,s=5)
    ax.set(xlabel='Number of Paths',ylabel='Error (Incoherent-Measured) [dB]',title='Error vs. Number of Paths'); ax.grid(True)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"error_vs_num_paths.png"),dpi=150); plt.show()

print("\n Cell 16 complete")

## CELL 17 · RSSI Metrics (Best, Incoherent, Coherent) vs Measured RSSI

In [ ]:
# ====================================================================
# CELL 17 — RSSI METRICS (best, incoherent, coherent) vs MEASURED RSSI
# ====================================================================
import pandas as pd, numpy as np, matplotlib.pyplot as plt, glob, os
from scipy.stats import spearmanr

print("=" * 70); print("CELL 17 — RSSI METRICS (Best, Incoherent, Coherent)"); print("=" * 70)
os.makedirs(OUT_DIR, exist_ok=True)

if 'df_sim' not in locals():
    summary_files=glob.glob(os.path.join(OUT_DIR,"path_solver_summary_*.csv"))
    if not summary_files: raise FileNotFoundError("No PathSolver summary CSV found. Run Cell 9b first.")
    sim_csv=max(summary_files,key=os.path.getctime)
    df_sim=pd.read_csv(sim_csv).dropna(subset=['path_loss_incoherent_db']).copy()
    print(f"Loaded simulation summary: {os.path.basename(sim_csv)}")
else:
    print("Reusing existing df_sim from previous cells.")

df_meas=pd.read_csv(MEASUREMENT_CSV)
print(f"\nMeasurement file columns: {list(df_meas.columns)}")
name_col=None
for candidate in ['name','receiver','receiver_name','rx_name','point']:
    if candidate in df_meas.columns: name_col=candidate; break
if name_col is None: raise KeyError("Measurement file has no column for receiver names.")
print(f"Using '{name_col}' as receiver name column.")

rssi_col=None
for candidate in ['local_measurement_dBm','rssi_dBm','RSSI_dBm','rx_power_dBm']:
    if candidate in df_meas.columns: rssi_col=candidate; break
if rssi_col is not None:
    df_meas['measured_rssi_dbm']=df_meas[rssi_col]; print(f"Using RSSI column: '{rssi_col}'")
elif 'path_loss_dB' in df_meas.columns:
    df_meas['measured_rssi_dbm']=EIRP_DBM-df_meas['path_loss_dB']
    print(f"Computed RSSI from 'path_loss_dB' using EIRP={EIRP_DBM} dBm")
else:
    raise KeyError("Measurement file must contain an RSSI column or 'path_loss_dB' column.")

df_meas=df_meas[[name_col,'measured_rssi_dbm']].copy()
df_meas=df_meas.rename(columns={name_col:'receiver'}).dropna(subset=['measured_rssi_dbm'])
print(f"Measurement data ready: {len(df_meas)} receivers")

df_merged=pd.merge(df_sim, df_meas, on='receiver', how='inner')
print(f"Merged: {len(df_merged)} matched receivers")
if len(df_merged)==0:
    print("\nNo matching receiver names!")
    print("Simulation receivers sample:", df_sim['receiver'].head().tolist())
    print("Measurement receivers sample:", df_meas['receiver'].head().tolist())
    raise ValueError("Merge resulted in zero rows.")

rssi_cols={'best_rssi':'rssi_best_dbm','incoherent_rssi':'rssi_incoherent_dbm','coherent_rssi':'rssi_coherent_dbm'}
missing=[name for name,col in rssi_cols.items() if col not in df_merged.columns]
if missing: raise KeyError(f"Simulated RSSI columns missing: {missing}. Re-run Cell 9b with SAVE_PER_RAY=True.")

def compute_metrics(sim_vals, meas_vals):
    sim=np.array(sim_vals); meas=np.array(meas_vals); valid=~np.isnan(sim)&~np.isnan(meas)
    sim=sim[valid]; meas=meas[valid]
    if len(sim)<2: return {'RMSE':np.nan,'MAE':np.nan,'MSE':np.nan,'R2':np.nan,'Bias':np.nan,'Spearman_rho':np.nan,'Spearman_p':np.nan}
    rmse=np.sqrt(np.mean((sim-meas)**2)); mae=np.mean(np.abs(sim-meas)); mse=np.mean((sim-meas)**2)
    ss_res=np.sum((sim-meas)**2); ss_tot=np.sum((meas-np.mean(meas))**2)
    r2=1-(ss_res/ss_tot) if ss_tot!=0 else 0; bias=np.mean(sim-meas)
    try: rho,p=spearmanr(meas,sim)
    except: rho,p=np.nan,np.nan
    return {'RMSE':rmse,'MAE':mae,'MSE':mse,'R2':r2,'Bias':bias,'Spearman_rho':rho,'Spearman_p':p}

rssi_metrics={}
for name,col in rssi_cols.items():
    rssi_metrics[name]=compute_metrics(df_merged[col].values, df_merged['measured_rssi_dbm'].values)

print("\n" + "=" * 80); print("RSSI METRICS (Simulated vs Measured RSSI)"); print("=" * 80)
print(f"{'Type':<18} {'RMSE':>8} {'MAE':>8} {'MSE':>8} {'R2':>7} {'Bias':>8} {'Spearman':>10}")
for name,m in rssi_metrics.items():
    print(f"{name:<18} {m['RMSE']:>8.2f} {m['MAE']:>8.2f} {m['MSE']:>8.2f} {m['R2']:>7.3f} {m['Bias']:>8.2f} {m['Spearman_rho']:>10.3f}")

df_rssi_metrics=pd.DataFrame(rssi_metrics).T.reset_index().rename(columns={'index':'Type'})
rssi_csv=os.path.join(OUT_DIR,"rssi_metrics.csv"); df_rssi_metrics.to_csv(rssi_csv,index=False)
print(f"\nSaved RSSI metrics to {rssi_csv}")

pathloss_metrics={}
for pl_type in ['best','incoherent','coherent']:
    col=f'path_loss_{pl_type}_db'
    if col in df_merged.columns and 'measured_pl_db' in df_merged.columns:
        pathloss_metrics[pl_type]=compute_metrics(df_merged[col].values, df_merged['measured_pl_db'].values)

if pathloss_metrics:
    x_labels=[f"PL_{t}" for t in pathloss_metrics]+[f"RSSI_{t}" for t in rssi_metrics]
    rmse_vals=[pathloss_metrics[t]['RMSE'] for t in pathloss_metrics]+[rssi_metrics[t]['RMSE'] for t in rssi_metrics]
    bias_vals=[pathloss_metrics[t]['Bias'] for t in pathloss_metrics]+[rssi_metrics[t]['Bias'] for t in rssi_metrics]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5))
    ax1.bar(x_labels,rmse_vals,color='steelblue'); ax1.set(ylabel='RMSE (dB)',title='RMSE Comparison'); ax1.tick_params(axis='x',rotation=45)
    ax2.bar(x_labels,bias_vals,color='orange'); ax2.axhline(0,color='k',linestyle='--'); ax2.set(ylabel='Bias (Sim-Meas) [dB]',title='Bias Comparison'); ax2.tick_params(axis='x',rotation=45)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"rssi_vs_pathloss_metrics.png"),dpi=150); plt.show()

print("\n Cell 17 complete")

## CELL 18 · Distance-Binned Analysis: RSSI, Path Loss, Number of Paths, Constructive/Destructive

In [ ]:
# ====================================================================
# CELL 18 — DISTANCE‑BINNED ANALYSIS: RSSI, PATH LOSS, NUMBER OF PATHS, CONSTRUCTIVE/DESTRUCTIVE
# ====================================================================
import pandas as pd, numpy as np, matplotlib.pyplot as plt, glob, os
from scipy.stats import spearmanr

print("=" * 70); print("CELL 18 — DISTANCE‑BINNED ANALYSIS"); print("=" * 70)
os.makedirs(OUT_DIR, exist_ok=True)

if 'df_merged' not in locals():
    summary_files=glob.glob(os.path.join(OUT_DIR,"path_solver_summary_*.csv"))
    if not summary_files: raise FileNotFoundError("No PathSolver summary CSV found. Run Cell 9b first.")
    sim_csv=max(summary_files,key=os.path.getctime)
    df_sim=pd.read_csv(sim_csv).dropna(subset=['path_loss_incoherent_db']).copy()
    df_meas=pd.read_csv(MEASUREMENT_CSV)
    for col in ['name','local_measurement_dBm','path_loss_dB']:
        if col not in df_meas.columns: raise KeyError(f"Measurement file missing column '{col}'.")
    df_meas=df_meas[['name','local_measurement_dBm','path_loss_dB']].dropna()
    df_meas.columns=['receiver','measured_rssi_dbm','measured_pl_db_original']
    df_meas['measured_pl_db']=EIRP_DBM-df_meas['measured_rssi_dbm']+SYS_GAIN
    df_merged=pd.merge(df_sim, df_meas, on='receiver', how='inner')
    print(f"Loaded {len(df_merged)} matched receivers")
else:
    print("Using existing df_merged.")
    if 'measured_pl_db' not in df_merged.columns and 'measured_rssi_dbm' in df_merged.columns:
        df_merged['measured_pl_db']=EIRP_DBM-df_merged['measured_rssi_dbm']+SYS_GAIN
        print("Recomputed measured_pl_db from measured_rssi_dbm.")
    elif 'measured_pl_db' not in df_merged.columns:
        raise KeyError("'measured_pl_db' column missing and cannot be recomputed.")

for col in ['dist_from_tx_m','num_paths','rssi_best_dbm','rssi_incoherent_dbm','rssi_coherent_dbm',
            'path_loss_best_db','path_loss_incoherent_db','path_loss_coherent_db','measured_rssi_dbm','measured_pl_db']:
    if col not in df_merged.columns:
        raise KeyError(f"Column '{col}' missing. Re-run Cell 9b with SAVE_PER_RAY=True.")

def compute_metrics(sim, meas):
    sim=np.array(sim); meas=np.array(meas); valid=~np.isnan(sim)&~np.isnan(meas)
    sim=sim[valid]; meas=meas[valid]
    if len(sim)<2: return {'RMSE':np.nan,'MAE':np.nan,'R2':np.nan,'Bias':np.nan,'Spearman':np.nan}
    rmse=np.sqrt(np.mean((sim-meas)**2)); mae=np.mean(np.abs(sim-meas))
    ss_res=np.sum((sim-meas)**2); ss_tot=np.sum((meas-np.mean(meas))**2)
    r2=1-(ss_res/ss_tot) if ss_tot!=0 else 0; bias=np.mean(sim-meas)
    try: rho,_=spearmanr(meas,sim)
    except: rho=np.nan
    return {'RMSE':rmse,'MAE':mae,'R2':r2,'Bias':bias,'Spearman':rho}

bin_width=250; max_dist=df_merged['dist_from_tx_m'].max()
bins=np.arange(0,max_dist+bin_width,bin_width)
labels=[f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]
df_merged['dist_bin']=pd.cut(df_merged['dist_from_tx_m'],bins=bins,labels=labels)

bin_stats=[]
for bin_label in labels:
    bin_data=df_merged[df_merged['dist_bin']==bin_label]
    if len(bin_data)==0: continue
    cp=bin_data['path_loss_coherent_db'].values; ip=bin_data['path_loss_incoherent_db'].values
    valid=~np.isnan(cp)&~np.isnan(ip)
    if np.sum(valid)>0:
        diff_db=cp[valid]-ip[valid]; constructive_fraction=np.mean(diff_db<0); mean_diff=np.mean(diff_db)
    else:
        constructive_fraction=np.nan; mean_diff=np.nan
    row={'dist_bin':bin_label,'avg_dist_m':bin_data['dist_from_tx_m'].mean(),
         'n_receivers':len(bin_data),'avg_num_paths':bin_data['num_paths'].mean(),
         'constructive_fraction':constructive_fraction,'coherent_minus_incoherent_db':mean_diff}
    for tag,col in [('rssi_best','rssi_best_dbm'),('rssi_incoherent','rssi_incoherent_dbm'),('rssi_coherent','rssi_coherent_dbm')]:
        m=compute_metrics(bin_data[col].values,bin_data['measured_rssi_dbm'].values)
        row[f'{tag}_RMSE']=m['RMSE']; row[f'{tag}_bias']=m['Bias']
    for tag,col in [('pl_best','path_loss_best_db'),('pl_incoherent','path_loss_incoherent_db'),('pl_coherent','path_loss_coherent_db')]:
        m=compute_metrics(bin_data[col].values,bin_data['measured_pl_db'].values)
        row[f'{tag}_RMSE']=m['RMSE']; row[f'{tag}_bias']=m['Bias']
    bin_stats.append(row)
df_bin=pd.DataFrame(bin_stats)
print("\nBin statistics (first 10 rows):"); print(df_bin.head(10).round(2).to_string(index=False))
bin_csv=os.path.join(OUT_DIR,"distance_bin_comprehensive.csv"); df_bin.to_csv(bin_csv,index=False)
print(f"\nSaved comprehensive bin statistics to {bin_csv}")

fig,axes=plt.subplots(2,3,figsize=(18,10))
for tag in ['rssi_best','rssi_incoherent','rssi_coherent']:
    axes[0,0].plot(df_bin['avg_dist_m'],df_bin[f'{tag}_RMSE'],'o-',label=tag.replace('rssi_',''))
axes[0,0].set(xlabel='Distance (m)',ylabel='RMSE (dB)',title='RSSI RMSE vs Distance'); axes[0,0].legend(); axes[0,0].grid(True)
for tag in ['rssi_best','rssi_incoherent','rssi_coherent']:
    axes[0,1].plot(df_bin['avg_dist_m'],df_bin[f'{tag}_bias'],'o-',label=tag.replace('rssi_',''))
axes[0,1].axhline(0,color='k',linestyle='--'); axes[0,1].set(xlabel='Distance (m)',ylabel='Bias [dB]',title='RSSI Bias vs Distance'); axes[0,1].legend(); axes[0,1].grid(True)
for tag in ['pl_best','pl_incoherent','pl_coherent']:
    axes[0,2].plot(df_bin['avg_dist_m'],df_bin[f'{tag}_RMSE'],'o-',label=tag.replace('pl_',''))
axes[0,2].set(xlabel='Distance (m)',ylabel='RMSE (dB)',title='Path Loss RMSE vs Distance'); axes[0,2].legend(); axes[0,2].grid(True)
for tag in ['pl_best','pl_incoherent','pl_coherent']:
    axes[1,0].plot(df_bin['avg_dist_m'],df_bin[f'{tag}_bias'],'o-',label=tag.replace('pl_',''))
axes[1,0].axhline(0,color='k',linestyle='--'); axes[1,0].set(xlabel='Distance (m)',ylabel='Bias [dB]',title='Path Loss Bias vs Distance'); axes[1,0].legend(); axes[1,0].grid(True)
axes[1,1].bar(df_bin['avg_dist_m'],df_bin['avg_num_paths'],width=150,alpha=0.7)
axes[1,1].set(xlabel='Distance (m)',ylabel='Average Number of Paths',title='Multipath Count vs Distance'); axes[1,1].grid(True)
axes[1,2].plot(df_bin['avg_dist_m'],df_bin['constructive_fraction'],'o-',color='green',label='Fraction constructive')
axes[1,2].plot(df_bin['avg_dist_m'],df_bin['coherent_minus_incoherent_db'],'s-',color='purple',label='Coherent-Incoherent (dB)')
axes[1,2].axhline(0,color='k',linestyle='--'); axes[1,2].set(xlabel='Distance (m)',ylabel='Fraction / dB',title='Constructive/Destructive Interference'); axes[1,2].legend(); axes[1,2].grid(True)
plt.tight_layout()
out_plot=os.path.join(OUT_DIR,"distance_bin_comprehensive.png"); plt.savefig(out_plot,dpi=150); plt.show()
print(f"Saved comprehensive plot to {out_plot}")
print("\n Cell 18 complete")

## CELL 19 · Building Density Analysis by Distance from TX

In [ ]:
# ====================================================================
# CELL 19 BUILDING DENSITY ANALYSIS BY DISTANCE FROM TX (with export)
# ====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os

def _safe(v):
    return float(v.item()) if hasattr(v,'item') else float(v)

tx=list(scene.transmitters.values())[0]
tx_pos=np.array([_safe(tx.position[0]),_safe(tx.position[1]),_safe(tx.position[2])])
print("BUILDING DENSITY BY DISTANCE FROM TX"); print("=" * 60)

rings=[(0,250),(250,500),(500,750),(750,1000),(1000,1500),(1500,2000),(2000,3000),(3000,5000)]
results=[]
for lo,hi in rings:
    heights=[]
    for obj in scene.objects.values():
        try:
            bbox=obj.mi_mesh.bbox()
            cx=(float(bbox.min[0])+float(bbox.max[0]))/2; cy=(float(bbox.min[1])+float(bbox.max[1]))/2
            d=np.sqrt((cx-tx_pos[0])**2+(cy-tx_pos[1])**2)
            h=float(bbox.max[2])-float(bbox.min[2])
            if lo<d<=hi and h>2.0: heights.append(h)
        except: pass
    n=len(heights); area_km2=np.pi*(hi**2-lo**2)/1e6
    density=n/area_km2 if area_km2>0 else 0; avg_h=float(np.mean(heights)) if n>0 else 0
    flag="OK" if density>10 else ("sparse" if density>2 else "EMPTY")
    results.append({'ring_start_m':lo,'ring_end_m':hi,'num_buildings':n,'area_km2':area_km2,
                    'density_per_km2':density,'avg_height_m':avg_h,'status':flag})
    print(f"  {lo:>5}-{hi:<5}m  N={n:>5}  density={density:>7.1f}/km2  avg_h={avg_h:>5.1f}m  {flag}")

df_density=pd.DataFrame(results)
out_csv=os.path.join(OUT_DIR,"building_density_by_distance.csv"); df_density.to_csv(out_csv,index=False)
print(f"\nSaved building density data to {out_csv}")

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
rings_center=[(lo+hi)/2 for lo,hi in rings]
ax1.bar(rings_center,df_density['density_per_km2'],width=200,alpha=0.7,color='steelblue')
ax1.set(xlabel='Distance from TX (m)',ylabel='Building Density (buildings/km2)',title='Building Density vs Distance'); ax1.grid(True,alpha=0.3)
ax2.bar(rings_center,df_density['avg_height_m'],width=200,alpha=0.7,color='forestgreen')
ax2.set(xlabel='Distance from TX (m)',ylabel='Average Building Height (m)',title='Average Height vs Distance'); ax2.grid(True,alpha=0.3)
plt.tight_layout()
out_plot=os.path.join(OUT_DIR,"building_density_analysis.png"); plt.savefig(out_plot,dpi=150); plt.show()
print(f"Saved plot to {out_plot}")
print("\n Building density analysis complete.")

## CELL 20 · Visualise Terrain Elevation

In [ ]:
# ====================================================================
# CELL 20 — VISUALISE TERRAIN ELEVATION (using global DEM_TIFF)
# ====================================================================
import sys, rasterio, matplotlib.pyplot as plt, os

log_file_path=os.path.join(OUT_DIR,"terrain_plot.log")
class Tee:
    def __init__(self,file,terminal): self.file=file; self.terminal=terminal
    def write(self,message): self.terminal.write(message); self.file.write(message)
    def flush(self): self.terminal.flush(); self.file.flush()

log_file=open(log_file_path,'w'); original_stdout=sys.stdout; sys.stdout=Tee(log_file,original_stdout)
print("=" * 70); print("TERRAIN ELEVATION MAP"); print("=" * 70)
print(f"DEM file: {DEM_TIFF}"); print(f"Output directory: {OUT_DIR}")

if not os.path.exists(DEM_TIFF): raise FileNotFoundError(f"DEM file not found: {DEM_TIFF}")

with rasterio.open(DEM_TIFF) as src:
    print(f"Shape: {src.shape}"); print(f"CRS: {src.crs}"); print(f"Bounds: {src.bounds}")
    elev=src.read(1)
    plt.figure(figsize=(10,8))
    plt.imshow(elev,cmap='terrain'); plt.colorbar(label='Elevation (m)')
    plt.title(f"Terrain Elevation – {os.path.basename(DEM_TIFF)}")
    out_png=os.path.join(OUT_DIR,"ground_floor_terrain.png")
    plt.savefig(out_png,dpi=150,bbox_inches='tight')
    print(f"Plot saved as: {out_png}"); plt.show()

sys.stdout=original_stdout; log_file.close()
print(f"Console output also saved to: {log_file_path}")
print("\n Terrain visualisation complete.")

## CELL 21 · Building Height Map Visualisation & Statistics

In [ ]:
# ====================================================================
# CELL 21 — BUILDING HEIGHT MAP VISUALISATION & STATISTICS
# ====================================================================
import numpy as np, matplotlib.pyplot as plt, os

os.makedirs(OUT_DIR, exist_ok=True)
if 'BUILDING_HEIGHT_MAP' not in locals() or not os.path.isfile(BUILDING_HEIGHT_MAP):
    BUILDING_HEIGHT_MAP="/home/georgeskai/Documents/Region/nottingham3602/nottingham_scene3602/2D_Building_Height_Map.npy"
    if not os.path.exists(BUILDING_HEIGHT_MAP):
        raise FileNotFoundError(f"Building height map not found at {BUILDING_HEIGHT_MAP}")

print("=" * 70); print("BUILDING HEIGHT MAP ANALYSIS"); print("=" * 70)
print(f"Loading height map from: {BUILDING_HEIGHT_MAP}")

height_map=np.load(BUILDING_HEIGHT_MAP)
plt.figure(figsize=(12,10))
plt.imshow(height_map,cmap='hot',interpolation='nearest')
plt.colorbar(label='Building Height (m)')
plt.title(f'Building Height Map\n{np.sum(height_map>0):,} Buildings')
plt.xlabel('X (pixels)'); plt.ylabel('Y (pixels)')
out_png=os.path.join(OUT_DIR,"building_height_map.png")
plt.savefig(out_png,dpi=150,bbox_inches='tight'); print(f"Plot saved as: {out_png}"); plt.show()

building_heights=height_map[height_map>0]
print(f"\nBuilding Statistics:")
print(f"  Total buildings: {len(building_heights)}")
print(f"  Mean height: {building_heights.mean():.2f} m")
print(f"  Median height: {np.median(building_heights):.2f} m")
print(f"  Max height: {building_heights.max():.2f} m")
print(f"  Min height: {building_heights.min():.2f} m")
print("\n Building height map analysis complete.")

## DIAG_5 · Samples-per-src Sufficiency Check

In [ ]:
# ====================================================================
# DIAGNOSTIC STEP 5 — Samples per src sufficiency check
# ====================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, glob, os

sim_csv=max(glob.glob(os.path.join(OUT_DIR,"path_solver_summary_*.csv")),key=os.path.getctime)
df_sim=pd.read_csv(sim_csv).dropna(subset=['path_loss_incoherent_db'])
df_meas=pd.read_csv(MEASUREMENT_CSV)
df=pd.merge(df_sim,df_meas,left_on='receiver',right_on='name',how='inner')
df['error']=df['path_loss_incoherent_db']-df['path_loss_dB']

tx=list(scene.transmitters.values())[0]
tx_pos=np.array([_safe(tx.position[0]),_safe(tx.position[1]),_safe(tx.position[2])])
df['dist_m']=np.linalg.norm(df[['x_m','y_m','z_m']].values-tx_pos,axis=1)

print("=" * 60); print("STEP 5 — NUM_SAMPLES SUFFICIENCY"); print("=" * 60)

SAMPLES=NUM_SAMPLES_PS; R_RX=0.5
print(f"\n[THEORETICAL RAY HITS AT DISTANCE d]")
print(f"  num_samples = {SAMPLES:,}  receiver radius = {R_RX} m")
print(f"\n  {'Distance':>10}  {'Expected hits':>15}  {'Sufficient?':>12}")
print(f"  {'-'*42}")
for d in [100,250,500,1000,2000,3000,5000]:
    hits=SAMPLES*(R_RX**2)/(4*d**2)
    ok="yes" if hits>=10 else ("marginal" if hits>=1 else "NO")
    print(f"  {d:>8}m  {hits:>15.2f}  {ok:>12}")

MIN_HITS=50
print(f"\n[REQUIRED SAMPLES FOR {MIN_HITS}+ HITS]")
for d in [500,1000,2000,3000,5000]:
    needed=MIN_HITS*(4*d**2)/(R_RX**2)
    print(f"  At {d:>5}m : need {needed:>15,.0f} samples")

print(f"\n[ACTUAL MEDIAN PATHS vs DISTANCE]")
bins=[0,100,250,500,1000,2000,3000,5000,9999]
labels=['<100m','100-250m','250-500m','500-1km','1-2km','2-3km','3-5km','>5km']
for lo,hi,lbl in zip(bins[:-1],bins[1:],labels):
    mask=df['dist_m'].between(lo,hi)
    if mask.sum()>0:
        med=df.loc[mask,'num_paths'].median(); mn=df.loc[mask,'num_paths'].min(); mx=df.loc[mask,'num_paths'].max()
        r=np.sqrt((df.loc[mask,'error']**2).mean()); flag="LOW" if med<10 else ("MARGINAL" if med<100 else "OK")
        print(f"  {lbl:<12} N={mask.sum():>4}  paths: med={med:>6.0f} min={mn:>4} max={mx:>6}  RMSE={r:>5.1f} dB  {flag}")

max_dist=df['dist_m'].max(); needed_for_max=MIN_HITS*(4*max_dist**2)/(R_RX**2)
print(f"\n[SUMMARY]")
print(f"  Max receiver distance : {max_dist:.0f} m")
print(f"  Samples needed for reliable coverage at max dist: {needed_for_max:,.0f}")
print(f"  Current num_samples                            : {SAMPLES:,}")
print(f"  Shortfall factor                               : {needed_for_max/SAMPLES:.0f}x")

print(f"\n[RMSE IF WE EXCLUDE UNDER-SAMPLED RECEIVERS]")
for max_d in [500,1000,1500,2000]:
    mask=df['dist_m']<=max_d
    if mask.sum()>5:
        r=np.sqrt((df.loc[mask,'error']**2).mean()); b=df.loc[mask,'error'].mean()
        print(f"  Keep only <={max_d:>4}m : N={mask.sum():>4}  RMSE={r:.1f} dB  bias={b:+.1f} dB")

fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(df['dist_m'],df['num_paths'],s=4,alpha=0.4,color='steelblue')
axes[0].axhline(10,color='red',linestyle='--',lw=1.5,label='min reliable (10)')
axes[0].axhline(100,color='orange',linestyle='--',lw=1.5,label='good (100)')
axes[0].set(xlabel='Distance from TX (m)',ylabel='Num paths (Sionna)',title='Num Paths vs Distance')
axes[0].set_yscale('log'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
colors=['green' if n>=100 else ('orange' if n>=10 else 'red') for n in df['num_paths']]
axes[1].scatter(df['dist_m'],df['error'],c=colors,s=5,alpha=0.5)
axes[1].axhline(0,color='black',linestyle='--',lw=1.5)
axes[1].set(xlabel='Distance from TX (m)',ylabel='Error: Sim-Meas (dB)',
            title='Error vs Distance\n(green=well-sampled, orange=marginal, red=under-sampled)')
axes[1].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,"diagnostic_step5.png"),dpi=150,bbox_inches='tight'); plt.show()
print("\n" + "=" * 60); print("Paste full output — especially 'RMSE if we exclude' section"); print("=" * 60)

## DIAG_6 · Bias Source Identification

In [ ]:
# ====================================================================
# DIAGNOSTIC STEP 6 — Bias source identification
# ====================================================================
import numpy as np, pandas as pd, glob, os

sim_csv=max(glob.glob(os.path.join(OUT_DIR,"path_solver_summary_*.csv")),key=os.path.getctime)
df_sim=pd.read_csv(sim_csv).dropna(subset=['path_loss_incoherent_db'])
df_meas=pd.read_csv(MEASUREMENT_CSV)
df=pd.merge(df_sim,df_meas,left_on='receiver',right_on='name',how='inner')
df['error']=df['path_loss_incoherent_db']-df['path_loss_dB']

tx=list(scene.transmitters.values())[0]
tx_pos=np.array([_safe(tx.position[0]),_safe(tx.position[1]),_safe(tx.position[2])])
df['dist_m']=np.linalg.norm(df[['x_m','y_m','z_m']].values-tx_pos,axis=1)

df_good=df[df['dist_m'].between(100,500)].copy()
print("=" * 60); print("STEP 6 — BIAS SOURCE (100-500m zone only)"); print("=" * 60)
print(f"\n[WELL-SAMPLED ZONE 100-500m]")
print(f"  N pairs   : {len(df_good)}")
print(f"  Bias      : {df_good['error'].mean():+.2f} dB")
print(f"  RMSE      : {np.sqrt((df_good['error']**2).mean()):.2f} dB")
print(f"  Std error : {df_good['error'].std():.2f} dB")

tx_pwr=EIRP_DBM
print(f"\n[TX EIRP]")
print(f"  EIRP_DBM in config    : {tx_pwr:.2f} dBm")
print(f"  Implied meas TX power : 70.00 dBm  (from Step 1)")
print(f"  Difference            : {tx_pwr-70.0:+.2f} dB")

print(f"\n[FREE-SPACE PATH LOSS CHECK at 300m]")
d=300.0; f=FREQUENCY_HZ; c_light=3e8
fspl=20*np.log10(4*np.pi*d*f/c_light)
near=df[df['dist_m'].between(280,320)]
if len(near)>0:
    sim_mean=near['path_loss_incoherent_db'].mean(); meas_mean=near['path_loss_dB'].mean()
    print(f"  Free-space PL at 300m, {f/1e9:.1f}GHz : {fspl:.2f} dB")
    print(f"  Sionna mean PL (280-320m)          : {sim_mean:.2f} dB")
    print(f"  Measured mean PL (280-320m)         : {meas_mean:.2f} dB")
    print(f"  Sionna vs FSPL                      : {sim_mean-fspl:+.2f} dB  (expected +5 to +20 for urban)")
    print(f"  Measured vs FSPL                    : {meas_mean-fspl:+.2f} dB")

print(f"\n[ANTENNA PATTERN CHECK]")
print(f"  TX pattern : {scene.tx_array.antenna.pattern.__class__.__name__}")
print(f"  RX pattern : {scene.rx_array.antenna.pattern.__class__.__name__}")
print(f"  TX rows x cols : {scene.tx_array.num_rows}x{scene.tx_array.num_cols}")
print(f"  RX rows x cols : {scene.rx_array.num_rows}x{scene.rx_array.num_cols}")

print(f"\n[MATERIAL SCATTERING COEFFICIENTS]")
for name,mat in list(scene.radio_materials.items())[:8]:
    try:
        sc=float(mat.scattering_coefficient); er=float(mat.relative_permittivity); si=float(mat.conductivity)
        print(f"  {name:<25} er={er:.2f}  sigma={si:.4f}  scatter={sc:.2f}")
    except:
        print(f"  {name:<25} (could not read properties)")

print(f"\n[BIAS INTERPRETATION]")
bias=df_good['error'].mean()
print(f"  Bias = {bias:+.2f} dB means simulation UNDER-predicts path loss")
print(f"  i.e. simulation thinks signal is {abs(bias):.1f} dB STRONGER than measured")
print(f"\n  Possible causes:")
print(f"  A) EIRP in sim ({tx_pwr:.1f} dBm) > effective EIRP in measurement")
print(f"  B) Antenna gain over-estimated (tr38901 on TX adds ~8 dBi vs iso)")
print(f"  C) Scattering coefficient too high (adds extra paths)")
print(f"  D) Material reflection too high (buildings too reflective)")

print(f"\n[TR38901 ANTENNA GAIN IMPACT]")
print(f"  tr38901 pattern peak gain : ~8 dBi")
print(f"  iso pattern gain          :  0 dBi")
print(f"  If TX uses tr38901 and meas assumes iso : +8 dB bias in sim")
print(f"  Your TX pattern           : {scene.tx_array.antenna.pattern.__class__.__name__}")
print(f"  Your RX pattern           : {scene.rx_array.antenna.pattern.__class__.__name__}")

print("\n" + "=" * 60); print("Paste full output — especially antenna patterns and materials"); print("=" * 60)